# EdgeNeXt pruning on Kaggle (training + packaging)

**Before running:** Settings (right panel) → Accelerator **GPU** (T4 or P100) and **Internet on** (needs a phone-verified account; used for pip, the pretrained weights and the datasets).

**Best way to run:** set the cell *Settings* below, then **Save Version → Save & Run All (Commit)**. It runs in the background (browser can be closed) for up to 12 h, and everything in `/kaggle/working` is saved as the version's output.

**Continue after a stop or for the next run:** *Add Input → Your Work → this notebook's output*. The *Restore* cell copies the earlier runs and `resume.pt` back, and `--resume` continues where it stopped. Runs that already have `log.json` are skipped.

Keep one commit under 12 h: EdgeNeXt-XXS (all three methods) fits in one; for EdgeNeXt-S run one method per commit.

`bench` is not run here (Kaggle is not an edge device).

In [ ]:
DATASET = 'cifar10'
ARCH    = 'edgenext_xx_small'
SEEDS   = [0]
BUDGET  = 0.5
METHODS = ['baseline', 'gated', 'depgraph']
EXTRA_GATED = ''
TAG     = ''
AMP     = ''
DO_TRAIN, DO_EXPORT = True, True
WORKERS = 4

In [ ]:
import os, glob, shutil, json, math, subprocess
WORK = '/kaggle/working/edgenext'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.total --format=csv
r = subprocess.run(['pip', 'download', '--no-deps', '-q', '-d', '/tmp/netcheck', 'six'], capture_output=True)
print('internet OK' if r.returncode == 0 else 'NO INTERNET: turn it on in Settings')

In [ ]:
for src in glob.glob('/kaggle/input/*/edgenext') + glob.glob('/kaggle/input/*/*/edgenext'):
    for sub in ('runs', 'artifacts', 'evidence', 'reports'):
        s = os.path.join(src, sub)
        if os.path.isdir(s):
            shutil.copytree(s, os.path.join(WORK, sub), dirs_exist_ok=True)
            print('restored', s)
!ls -R {WORK}/runs 2>/dev/null | head -40


In [ ]:
%%writefile edgenext_bench.py
"""
edgenext_bench.py - pruned vs. unpruned EdgeNeXt (timm), measured end to end.

Every number this script writes is measured in the run that writes it. Nothing
is typed in, estimated, copied from another run or generated at random. Each
results file names the device, runtime versions, thread count, the exact model
files (SHA-256) and the exact evaluation images it used, so every figure in a
report can be traced back to a measurement.

Subcommands
-----------
  export           (laptop, needs torch + timm) Package one experiment: both
                   models as PyTorch / ONNX / TFLite FP32 / TFLite INT8, the
                   evaluation images, and a manifest with checksums, parameter
                   counts and MACs (all computed, never typed).
  bench            (any device) Measure accuracy and latency of pruned AND
                   unpruned on every runtime available on this device. Verifies
                   file checksums against the manifest first.
  report           Combine results files from all devices into one Markdown /
                   CSV report with speedups, confidence intervals and
                   significance tests computed from the raw measurements.
  summary          Every run and seed at once (training log.json files +
                   results files): accuracy mean +- std, paired change vs the
                   baseline, complexity, training time, device metrics and
                   accuracy-vs-cost figures.
  depgraph-repro   Run Torch-Pruning (DepGraph) on timm EdgeNeXt and log what
                   happens (traceback, versions), as evidence for the paper.

Metrics recorded
----------------
  export   params, MACs/FLOPs, file size per format, ONNX/TFLite numerical
           checks, INT8 checks, TFLite weight and peak activation memory
           (tensor-arena estimate for microcontrollers)
  bench    top-1 and top-5 with 95% intervals, per-image predictions,
           batch-1 latency (mean, std, median, p5, p95, p99, min, max,
           throughput), model load time, first-inference (cold-start)
           latency, peak RAM in a fresh process, CPU temperature, frequency
           and Raspberry Pi throttling flags; with --sustained-seconds:
           sustained throughput and energy per inference (on-board sensor,
           or an external USB meter log given to report/summary --power-csv)
  report   speedup with bootstrap CI, McNemar test, runtime agreement

Training (unpruned baseline, gated pruning, DepGraph baseline) lives in
edgenext_gated_prune.py, which imports this file; keep both in one folder.

Typical order for one experiment (example: CIFAR-10, EdgeNeXt-XXS, seed 0)
-------------------------------------------------------------------------
  # 1. train on a GPU (edgenext_gated_prune.py)
  python edgenext_gated_prune.py baseline --arch edgenext_xx_small --dataset cifar10 \
      --data-root ./data --recipe xxs --seed 0 --out runs/c10_xxs/baseline_s0
  python edgenext_gated_prune.py gated --arch edgenext_xx_small --dataset cifar10 \
      --data-root ./data --recipe xxs --budget 0.5 --seed 0 --out runs/c10_xxs/gated_b50_s0

  # 2. package both models (laptop or the GPU machine)
  python edgenext_bench.py export --experiment c10_xxs_b50_s0 --dataset cifar10 \
      --data-root ./data --pruned runs/c10_xxs/gated_b50_s0/model.pt \
      --unpruned runs/c10_xxs/baseline_s0/model.pt --out artifacts/c10_xxs_b50_s0

  # 3. measure on every device (copy the artifacts folder; the script is inside)
  python edgenext_bench.py bench --artifacts artifacts/c10_xxs_b50_s0 --device-label laptop \
      --sustained-seconds 60
  python3 edgenext_bench.py bench --artifacts c10_xxs_b50_s0 --device-label rpi3bplus
  python3 edgenext_bench.py bench --artifacts c10_xxs_b50_s0 --device-label qemu-a72 --emulated yes

  # 4. one report per package, then one summary of every run
  python edgenext_bench.py report --results artifacts/c10_xxs_b50_s0/results/*.json \
      --out reports/c10_xxs_b50_s0
  python edgenext_bench.py summary --logs "runs/**/log.json" \
      --results "artifacts/*/results/*.json" --out reports/summary

Device-side requirements: numpy, pillow, plus onnxruntime and/or ai-edge-litert
(or tflite-runtime). torch/timm are only needed for the "pytorch" runtime,
export and depgraph-repro.
"""

import argparse
import csv
import glob
import hashlib
import json
import math
import os
import pickle
import platform
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone

import numpy as np

SCHEMA = "edgenext-bench/1"
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

TINY10_DEFAULT_WNIDS = [
    "n02124075", "n04067472", "n04540053", "n04099969", "n07749582",
    "n01641577", "n02802426", "n09246464", "n07920052", "n03970156",
]

DATASETS = {
    "cifar10": {"num_classes": 10, "eval_split": "test",
                "preprocess": {"mode": "resize", "size": 224}},
    "cifar100": {"num_classes": 100, "eval_split": "test",
                 "preprocess": {"mode": "resize", "size": 224}},
    "tiny10": {"num_classes": 10, "eval_split": "val",
               "preprocess": {"mode": "resize_short_center_crop", "resize": 256, "size": 224}},
    "tiny200": {"num_classes": 200, "eval_split": "val",
                "preprocess": {"mode": "resize_short_center_crop", "resize": 256, "size": 224}},
    "imagefolder": {"num_classes": None, "eval_split": "val",
                    "preprocess": {"mode": "presized", "resize": 256, "size": 224}},
}
IMG_EXTS = (".jpeg", ".jpg", ".png", ".bmp", ".webp")

ARCH_BY_STAGE4_DIM = {168: "edgenext_xx_small", 192: "edgenext_x_small",
                      304: "edgenext_small", 584: "edgenext_base"}

ALL_RUNTIMES = ["pytorch", "onnx", "tflite_fp32", "tflite_int8"]



def now_iso():
    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def file_record(path, base_dir):
    return {"file": os.path.relpath(path, base_dir), "sha256": sha256_file(path),
            "bytes": os.path.getsize(path)}


def write_json(path, obj):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def log(msg):
    print(msg, flush=True)


def pkg_version(name):
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return None


def _read_text(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception:
        return ""


def cpu_model_name():
    info = _read_text("/proc/cpuinfo")
    for key in ("model name", "Model", "Hardware", "CPU part"):
        for line in info.splitlines():
            if line.split(":")[0].strip() == key:
                return line.split(":", 1)[1].strip()
    if platform.system() == "Windows":
        return platform.processor() or None
    if platform.system() == "Darwin":
        try:
            return subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                                  capture_output=True, text=True, timeout=5).stdout.strip()
        except Exception:
            return None
    return platform.processor() or None


def looks_emulated(hints):
    """True only for instruction-set emulation (QEMU TCG / qemu-user). WSL, KVM and
    Hyper-V are virtualised but run native instructions, so they do not count."""
    return any("qemu" in h.lower() for h in hints)


def detect_emulation():
    """Best-effort hints only; the --emulated flag is authoritative."""
    hints = []
    try:
        r = subprocess.run(["systemd-detect-virt"], capture_output=True, text=True, timeout=5)
        v = r.stdout.strip()
        if v and v != "none":
            hints.append("systemd-detect-virt=" + v)
    except Exception:
        pass
    vendor = _read_text("/sys/class/dmi/id/sys_vendor").strip()
    if "QEMU" in vendor:
        hints.append("dmi sys_vendor=" + vendor)
    cpuinfo = _read_text("/proc/cpuinfo")
    if platform.machine() in ("aarch64", "arm64") and ("GenuineIntel" in cpuinfo or "AuthenticAMD" in cpuinfo):
        hints.append("aarch64 userland on x86 cpuinfo (qemu-user / binfmt)")
    if platform.machine() in ("aarch64", "arm64") and "QEMU" in cpuinfo:
        hints.append("qemu in /proc/cpuinfo")
    if os.environ.get("QEMU_CPU"):
        hints.append("QEMU_CPU=" + os.environ["QEMU_CPU"])
    return hints


def env_info():
    return {
        "hostname": socket.gethostname(),
        "os": platform.platform(),
        "machine": platform.machine(),
        "cpu_model": cpu_model_name(),
        "board_model": _read_text("/proc/device-tree/model").strip("\x00\n ") or None,
        "logical_cpus": os.cpu_count(),
        "python": platform.python_version(),
        "packages": {p: pkg_version(p) for p in (
            "numpy", "pillow", "torch", "timm", "onnx", "onnxruntime", "ai-edge-litert",
            "tflite-runtime", "tensorflow", "litert-torch", "ai-edge-torch",
            "ai-edge-quantizer", "torch-pruning")},
        "emulation_hints": detect_emulation(),
    }



def _proc_status_kib(key):
    for line in _read_text("/proc/self/status").splitlines():
        if line.startswith(key + ":"):
            try:
                return int(line.split()[1])
            except (IndexError, ValueError):
                return None
    return None


def rss_bytes():
    """Current resident memory of this process."""
    v = _proc_status_kib("VmRSS")
    if v is not None:
        return v * 1024
    try:
        import psutil
        return int(psutil.Process().memory_info().rss)
    except Exception:
        return None


def peak_rss_bytes():
    """Peak resident memory of this process since it started."""
    v = _proc_status_kib("VmHWM")
    if v is not None:
        return v * 1024
    try:
        import resource
        r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return int(r if sys.platform == "darwin" else r * 1024)
    except Exception:
        pass
    try:
        import psutil
        return int(getattr(psutil.Process().memory_info(), "peak_wset", 0)) or None
    except Exception:
        return None


_THROTTLE_BITS = {0: "under_voltage_now", 1: "freq_capped_now", 2: "throttled_now", 3: "soft_temp_limit_now",
                  16: "under_voltage_since_boot", 17: "freq_capped_since_boot", 18: "throttled_since_boot",
                  19: "soft_temp_limit_since_boot"}


def device_state():
    """CPU temperatures (sysfs thermal zones), current CPU frequencies and the
    Raspberry Pi throttling flags (vcgencmd get_throttled)."""
    st = {"unix_time": time.time(), "temps_c": {}, "max_temp_c": None, "cpu_mhz": None, "throttled": None}
    for z in sorted(glob.glob("/sys/class/thermal/thermal_zone*")):
        t = _read_text(os.path.join(z, "temp")).strip()
        if t.lstrip("-").isdigit():
            name = _read_text(os.path.join(z, "type")).strip() or os.path.basename(z)
            st["temps_c"][name] = int(t) / 1000.0
    if st["temps_c"]:
        st["max_temp_c"] = max(st["temps_c"].values())
    f = []
    for c in sorted(glob.glob("/sys/devices/system/cpu/cpu[0-9]*/cpufreq/scaling_cur_freq")):
        v = _read_text(c).strip()
        if v.isdigit():
            f.append(int(v) / 1000.0)
    st["cpu_mhz"] = f or None
    if shutil.which("vcgencmd"):
        try:
            out = subprocess.run(["vcgencmd", "get_throttled"], capture_output=True, text=True, timeout=3).stdout
            v = int(out.strip().split("=")[1], 16)
            st["throttled"] = dict({"raw": hex(v)}, **{n: bool(v >> b & 1) for b, n in _THROTTLE_BITS.items()})
        except Exception:
            pass
    return st


class PowerMeter:
    """Energy over a time window from an on-board sensor, if this device has one:
      rapl      Intel/AMD CPU package energy counters (native Linux; usually needs
                root or `sudo chmod a+r /sys/class/powercap/intel-rapl:*/energy_uj`)
      pi5_pmic  Raspberry Pi 5 PMIC rails via `vcgencmd pmic_read_adc`, sampled
                (SoC and memory rails; USB and other peripherals not included)
      ina3221   Jetson VDD_IN input-rail monitor (hwmon), sampled
    Raspberry Pi 3B+/4 and most laptops under WSL2 have none: use an external
    USB power meter with the window timestamps (see bench --sustained-seconds)."""

    def __init__(self):
        self.kind, self.detail, self.reason = None, None, "no on-board power sensor found"
        self._rapl, self._hw = [], None
        for d in sorted(glob.glob("/sys/class/powercap/intel-rapl:*")):
            if os.path.basename(d).count(":") != 1:
                continue
            e = os.path.join(d, "energy_uj")
            if not os.path.exists(e):
                continue
            if not os.access(e, os.R_OK) or not _read_text(e).strip().isdigit():
                self.reason = "RAPL present but %s is not readable (run as root or chmod a+r)" % e
                continue
            rng = _read_text(os.path.join(d, "max_energy_range_uj")).strip()
            self._rapl.append((e, int(rng) if rng.isdigit() else None,
                               _read_text(os.path.join(d, "name")).strip()))
        if self._rapl:
            self.kind = "rapl"
            self.detail = "CPU package energy counters: " + ", ".join(n for _, _, n in self._rapl)
            self.reason = None
            return
        if "Raspberry Pi 5" in _read_text("/proc/device-tree/model") and self._pi5_power() is not None:
            self.kind, self.detail, self.reason = "pi5_pmic", "Pi 5 PMIC rails (SoC + memory), sampled 5 Hz", None
            return
        for lbl in glob.glob("/sys/bus/i2c/drivers/ina3221*/*/hwmon/hwmon*/in*_label"):
            if "VDD_IN" in _read_text(lbl):
                n = os.path.basename(lbl)[2:-6]
                base = os.path.dirname(lbl)
                self._hw = (os.path.join(base, "in%s_input" % n), os.path.join(base, "curr%s_input" % n))
                if self._ina_power() is not None:
                    self.kind, self.detail, self.reason = "ina3221", "Jetson VDD_IN rail, sampled 5 Hz", None
                    return

    @staticmethod
    def _pi5_power():
        try:
            out = subprocess.run(["vcgencmd", "pmic_read_adc"], capture_output=True, text=True, timeout=3).stdout
        except Exception:
            return None
        amps, volts = {}, {}
        for line in out.splitlines():
            parts = line.split()
            if len(parts) < 2 or "=" not in parts[1]:
                continue
            name, val = parts[0], parts[1].split("=")[1]
            try:
                x = float(val.rstrip("AV"))
            except ValueError:
                continue
            if name.endswith("_A"):
                amps[name[:-2]] = x
            elif name.endswith("_V"):
                volts[name[:-2]] = x
        common = set(amps) & set(volts)
        return sum(amps[k] * volts[k] for k in common) if common else None

    def _ina_power(self):
        v, i = _read_text(self._hw[0]).strip(), _read_text(self._hw[1]).strip()
        if v.isdigit() and i.isdigit():
            return int(v) * int(i) / 1e6
        return None

    def start(self):
        if self.kind is None:
            return None
        if self.kind == "rapl":
            return {"t": time.perf_counter(), "e": [int(_read_text(p).strip()) for p, _, _ in self._rapl]}
        import threading
        tok = {"t": time.perf_counter(), "j": 0.0, "n": 0, "stop": threading.Event()}
        fn = self._pi5_power if self.kind == "pi5_pmic" else self._ina_power

        def loop():
            t_prev, p_prev = time.perf_counter(), fn()
            while not tok["stop"].wait(0.2):
                t, p = time.perf_counter(), fn()
                if p is not None and p_prev is not None:
                    tok["j"] += 0.5 * (p + p_prev) * (t - t_prev)
                    tok["n"] += 1
                t_prev, p_prev = t, p
        tok["thread"] = threading.Thread(target=loop, daemon=True)
        tok["thread"].start()
        return tok

    def stop(self, tok):
        if tok is None:
            return None
        if self.kind == "rapl":
            dt = time.perf_counter() - tok["t"]
            j = 0.0
            for (p, rng, _), e0 in zip(self._rapl, tok["e"]):
                d = int(_read_text(p).strip()) - e0
                if d < 0 and rng:
                    d += rng
                j += d / 1e6
            return {"joules": j, "seconds": dt, "avg_w": j / dt if dt > 0 else None, "samples": None}
        tok["stop"].set()
        tok["thread"].join()
        dt = time.perf_counter() - tok["t"]
        return {"joules": tok["j"], "seconds": dt, "avg_w": tok["j"] / dt if dt > 0 else None,
                "samples": tok["n"]}


def tflite_memory_estimate(path):
    """Weights and activation memory of a .tflite graph, from its tensors and the
    op execution order: peak = largest total size of activation tensors alive
    at the same time (no scratch buffers, no allocator overhead). An estimate of
    the TFLite Micro tensor arena; the firmware's arena_used_bytes() is the
    measured value."""
    Interpreter, _ = _tflite_interpreter_cls()
    kw = {}
    try:
        from ai_edge_litert.interpreter import OpResolverType
        kw["experimental_op_resolver_type"] = OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    except Exception:
        pass
    try:
        it = Interpreter(model_path=path, **kw)
    except TypeError:
        it = Interpreter(model_path=path)
    it.allocate_tensors()
    td = {t["index"]: t for t in it.get_tensor_details()}
    ops = it._get_ops_details()

    def nbytes(i):
        t = td.get(i)
        if t is None:
            return 0
        return int(np.prod(t["shape"])) * np.dtype(t["dtype"]).itemsize if len(t["shape"]) else \
            np.dtype(t["dtype"]).itemsize

    produced = {}
    last_use = {}
    for k, op in enumerate(ops):
        for i in op["inputs"]:
            if i >= 0:
                last_use[i] = k
        for o in op["outputs"]:
            produced[o] = k
    inputs = [d["index"] for d in it.get_input_details()]
    outputs = [d["index"] for d in it.get_output_details()]
    live = {}
    for i in inputs:
        live[i] = (-1, last_use.get(i, -1))
    for o, k in produced.items():
        live[o] = (k, len(ops) if o in outputs else max(last_use.get(o, k), k))
    peak, at = 0, None
    for k in range(len(ops)):
        s = sum(nbytes(i) for i, (a, b) in live.items() if a <= k <= b)
        if s > peak:
            peak, at = s, k
    weights = sum(nbytes(i) for i in last_use if i not in live)
    return {"peak_activation_bytes": int(peak), "peak_at_op": ops[at]["op_name"] if at is not None else None,
            "largest_activation_bytes": int(max((nbytes(i) for i in live), default=0)),
            "activation_tensors_total_bytes": int(sum(nbytes(i) for i in live)),
            "weights_bytes": int(weights), "ops": len(ops),
            "note": "liveness estimate over the op order; excludes scratch buffers and allocator overhead"}



def latency_summary(ms):
    a = np.asarray(ms, dtype=np.float64)
    return {
        "n": int(a.size), "mean": float(a.mean()), "std": float(a.std(ddof=1)) if a.size > 1 else 0.0,
        "median": float(np.median(a)), "p05": float(np.percentile(a, 5)),
        "p95": float(np.percentile(a, 95)), "p99": float(np.percentile(a, 99)),
        "min": float(a.min()), "max": float(a.max()),
        "throughput_img_s_from_mean": float(1000.0 / a.mean()),
        "throughput_img_s_from_median": float(1000.0 / np.median(a)),
    }


def wilson_interval(correct, n, z=1.959963984540054):
    if n == 0:
        return [float("nan"), float("nan")]
    p = correct / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return [max(0.0, centre - half), min(1.0, centre + half)]


def mcnemar_exact(correct_a, correct_b):
    """Two-sided exact McNemar test on paired per-image correctness."""
    a = np.asarray(correct_a, bool)
    b = np.asarray(correct_b, bool)
    n01 = int(np.sum(a & ~b))
    n10 = int(np.sum(~a & b))
    n = n01 + n10
    if n == 0:
        return {"a_only_correct": n01, "b_only_correct": n10, "p_value": 1.0}
    k = min(n01, n10)
    logs = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2)
            for i in range(k + 1)]
    m = max(logs)
    tail = math.exp(m) * sum(math.exp(x - m) for x in logs)
    return {"a_only_correct": n01, "b_only_correct": n10, "p_value": min(1.0, 2.0 * tail)}


def bootstrap_ratio_of_medians(num_ms, den_ms, iters=4000, seed=0):
    """CI for median(num)/median(den), resampling each sample independently."""
    rng = np.random.default_rng(seed)
    a = np.asarray(num_ms, np.float64)
    b = np.asarray(den_ms, np.float64)
    ra = np.median(a[rng.integers(0, a.size, (iters, a.size))], axis=1)
    rb = np.median(b[rng.integers(0, b.size, (iters, b.size))], axis=1)
    r = ra / rb
    return [float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))]



def _cifar_files(root, dataset, split):
    if dataset == "cifar10":
        base = os.path.join(root, "cifar-10-batches-py")
        names = ["test_batch"] if split == "test" else ["data_batch_%d" % i for i in range(1, 6)]
        key = b"labels"
    else:
        base = os.path.join(root, "cifar-100-python")
        names = ["test"] if split == "test" else ["train"]
        key = b"fine_labels"
    return [os.path.join(base, n) for n in names], key


def imagefolder_classes(root):
    d = os.path.join(root, "train")
    if not os.path.isdir(d):
        raise FileNotFoundError("expected %s/train/<class>/... and %s/val/<class>/..." % (root, root))
    return sorted(x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x)))


def list_imagefolder(root, split, per_class=None):
    """(paths, labels, classes) for <root>/<split>/<class>/<image>; class order = sorted train folders."""
    classes = imagefolder_classes(root)
    paths, labels = [], []
    for label, c in enumerate(classes):
        folder = os.path.join(root, split, c)
        if not os.path.isdir(folder):
            raise FileNotFoundError("missing %s" % folder)
        files = sorted(f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS))
        if per_class is not None:
            files = files[:per_class]
        paths += [os.path.join(folder, f) for f in files]
        labels += [label] * len(files)
    return paths, np.asarray(labels, np.int64), classes


def dataset_num_classes(dataset, root):
    n = DATASETS[dataset]["num_classes"]
    return n if n is not None else len(imagefolder_classes(root))


def _resize_short_center_crop(im, resize, size):
    """torchvision Resize(int) + CenterCrop(size) semantics on a PIL image (bilinear)."""
    from PIL import Image
    bilinear = getattr(Image, "Resampling", Image).BILINEAR
    w, h = im.size
    if w <= h:
        ow, oh = resize, int(resize * h / w)
    else:
        oh, ow = resize, int(resize * w / h)
    if (ow, oh) != (w, h):
        im = im.resize((ow, oh), bilinear)
    top = int(round((oh - size) / 2.0))
    left = int(round((ow - size) / 2.0))
    return im.crop((left, top, left + size, top + size))


def load_presized(path, cfg):
    """Open an image file and apply the fixed resize + centre-crop (uint8 HWC)."""
    from PIL import Image
    with Image.open(path) as im:
        return np.asarray(_resize_short_center_crop(im.convert("RGB"), int(cfg["resize"]), int(cfg["size"])),
                          np.uint8)


def load_raw_split(dataset, root, split, class_order=None, download=False, per_class=None):
    """Returns (images uint8 [N,H,W,3], labels int64 [N], class_names list).
    For imagefolder the images are already resized/cropped (they can be large),
    and per_class caps how many are loaded."""
    if dataset == "imagefolder":
        cfg = DATASETS["imagefolder"]["preprocess"]
        paths, labels, classes = list_imagefolder(root, split, per_class)
        if not paths:
            raise FileNotFoundError("no images under %s/%s" % (root, split))
        return np.stack([load_presized(p, cfg) for p in paths]), labels, classes
    if dataset in ("cifar10", "cifar100"):
        files, key = _cifar_files(root, dataset, split)
        if not all(os.path.exists(f) for f in files):
            if download:
                import torchvision
                cls = torchvision.datasets.CIFAR10 if dataset == "cifar10" else torchvision.datasets.CIFAR100
                cls(root=root, train=(split == "train"), download=True)
            else:
                raise FileNotFoundError(
                    "CIFAR files not found under %s (expected %s). Pass --download or point "
                    "--data-root at the folder that contains them." % (root, files[0]))
        imgs, labels = [], []
        for f in files:
            with open(f, "rb") as fh:
                d = pickle.load(fh, encoding="bytes")
            imgs.append(np.asarray(d[b"data"], np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1))
            labels.extend(d[key])
        meta = os.path.join(os.path.dirname(files[0]), "batches.meta" if dataset == "cifar10" else "meta")
        names = None
        if os.path.exists(meta):
            with open(meta, "rb") as fh:
                m = pickle.load(fh, encoding="bytes")
            raw = m.get(b"label_names") or m.get(b"fine_label_names")
            names = [x.decode() for x in raw] if raw else None
        imgs, labels = np.concatenate(imgs), np.asarray(labels, np.int64)
        if per_class is not None:
            keep = stratified_indices(labels, per_class=per_class, num_classes=DATASETS[dataset]["num_classes"])
            imgs, labels = imgs[keep], labels[keep]
        return np.ascontiguousarray(imgs), labels, names

    if dataset in ("tiny10", "tiny200"):
        from PIL import Image
        n_cls = DATASETS[dataset]["num_classes"]
        wnids = list(class_order) if class_order else None
        if wnids is None:
            wf = os.path.join(root, "wnids.txt")
            if os.path.exists(wf):
                with open(wf) as fh:
                    wnids = [l.strip() for l in fh if l.strip()][:n_cls]
            elif dataset == "tiny10":
                wnids = list(TINY10_DEFAULT_WNIDS)
            else:
                wnids = sorted(os.listdir(os.path.join(root, "train")))[:n_cls]
        if len(wnids) != n_cls:
            raise ValueError("%s needs %d classes, found %d" % (dataset, n_cls, len(wnids)))
        paths, labels = [], []
        for label, wnid in enumerate(wnids):
            if split == "train":
                cands = [os.path.join(root, "train", wnid, "images"), os.path.join(root, "train", wnid)]
                folder = next((c for c in cands if os.path.isdir(c)), None)
                if folder is None:
                    raise FileNotFoundError("No train folder for %s under %s" % (wnid, root))
                files = sorted(f for f in os.listdir(folder) if f.lower().endswith((".jpeg", ".jpg", ".png")))
                paths += [os.path.join(folder, f) for f in files]
                labels += [label] * len(files)
        if split == "val":
            ann = os.path.join(root, "val", "val_annotations.txt")
            if os.path.exists(ann):
                with open(ann) as fh:
                    rows = [l.split() for l in fh if l.strip()]
                idx = {w: i for i, w in enumerate(wnids)}
                rows = sorted((r[1], r[0]) for r in rows if r[1] in idx)
                for wnid, fn in rows:
                    paths.append(os.path.join(root, "val", "images", fn))
                    labels.append(idx[wnid])
            else:
                for label, wnid in enumerate(wnids):
                    folder = os.path.join(root, "val", wnid)
                    if not os.path.isdir(folder):
                        raise FileNotFoundError("No val folder for %s under %s" % (wnid, root))
                    files = sorted(f for f in os.listdir(folder) if f.lower().endswith((".jpeg", ".jpg", ".png")))
                    paths += [os.path.join(folder, f) for f in files]
                    labels += [label] * len(files)
        if not paths:
            raise FileNotFoundError("No %s images found for %s under %s" % (split, dataset, root))
        labels = np.asarray(labels, np.int64)
        if per_class is not None:
            keep = stratified_indices(labels, per_class=per_class, num_classes=n_cls)
            paths, labels = [paths[i] for i in keep], labels[keep]
        imgs = np.stack([np.asarray(Image.open(p).convert("RGB"), np.uint8) for p in paths])
        return imgs, labels, wnids

    raise ValueError("unknown dataset " + dataset)


def stratified_indices(labels, per_class=None, total=None, num_classes=None, spread=False):
    """Deterministic class-balanced subset. per_class wins over total.
    spread=False: first k of each class (in file order); spread=True: evenly spaced."""
    labels = np.asarray(labels)
    nc = num_classes or int(labels.max()) + 1
    if per_class is None:
        if total is None:
            return np.arange(labels.size)
        per_class = max(1, int(math.ceil(total / nc)))
    out = []
    for c in range(nc):
        idx = np.flatnonzero(labels == c)
        if idx.size == 0:
            continue
        if spread and idx.size > per_class:
            idx = idx[np.linspace(0, idx.size - 1, per_class).round().astype(int)]
        out.append(idx[:per_class])
    return np.sort(np.concatenate(out))


def preprocess(img_u8, cfg):
    """uint8 HWC -> float32 CHW, identical to the pruning scripts' eval transform
    (torchvision Resize on PIL images = PIL bilinear; ToTensor; ImageNet Normalize)."""
    from PIL import Image
    bilinear = getattr(Image, "Resampling", Image).BILINEAR
    size = int(cfg["size"])
    if cfg["mode"] == "presized":
        if img_u8.shape[:2] != (size, size):
            raise ValueError("presized image has shape %s, expected %dx%d" % (img_u8.shape, size, size))
        x = np.asarray(img_u8, np.float32) / 255.0
    else:
        im = Image.fromarray(img_u8)
        if cfg["mode"] == "resize":
            im = im.resize((size, size), bilinear)
        elif cfg["mode"] == "resize_short_center_crop":
            im = _resize_short_center_crop(im, int(cfg["resize"]), size)
        else:
            raise ValueError(cfg["mode"])
        x = np.asarray(im, np.float32) / 255.0
    x = (x - np.asarray(IMAGENET_MEAN, np.float32)) / np.asarray(IMAGENET_STD, np.float32)
    return np.ascontiguousarray(x.transpose(2, 0, 1))


def preprocess_batch(imgs_u8, cfg):
    return np.stack([preprocess(im, cfg) for im in imgs_u8]).astype(np.float32)



def install_xca_patch():
    """timm's CrossCovarianceAttn reshapes to the INPUT width after attention,
    which breaks once heads are pruned. The pruned checkpoints were pickled with
    instance forwards bound to `CrossCovarianceAttn.patched_forward`, so that
    attribute must exist before torch.load. For unpruned models the patched
    forward is numerically identical to timm's."""
    import torch.nn.functional as F
    from timm.models.edgenext import CrossCovarianceAttn

    def patched_forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, -1).permute(2, 0, 3, 4, 1)
        q, k, v = qkv.unbind(0)
        attn = (F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).permute(0, 3, 1, 2).reshape(B, N, -1)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    CrossCovarianceAttn.patched_forward = patched_forward
    CrossCovarianceAttn.forward = patched_forward


def install_sdta_patch():
    """timm's SplitTransposeBlock splits its input with x.chunk(n_convs + 1),
    which only works for the original widths. This forward splits with explicit
    sizes [w] * n_convs + [C - n_convs * w], w = width of the depthwise convs,
    so a block whose trunk was pruned (smaller w and/or smaller pass-through
    part) still runs. For unpruned models the sizes equal chunk()'s sizes, so
    the output is identical."""
    import torch
    from timm.models.edgenext import SplitTransposeBlock

    def split_forward(self, x):
        shortcut = x
        n = len(self.convs)
        w = int(self.convs[0].in_channels)
        c = int(self.norm_xca.normalized_shape[0])
        spx = torch.split(x, [w] * n + [c - w * n], dim=1)
        spo = []
        sp = spx[0]
        for i, conv in enumerate(self.convs):
            if i > 0:
                sp = sp + spx[i]
            sp = conv(sp)
            spo.append(sp)
        spo.append(spx[-1])
        x = torch.cat(spo, 1)

        B, C, H, W = x.shape
        x = x.reshape(B, C, H * W).permute(0, 2, 1)
        if self.pos_embd is not None:
            pos_encoding = self.pos_embd((B, H, W)).reshape(B, -1, x.shape[1]).permute(0, 2, 1)
            x = x + pos_encoding
        x = x + self.drop_path(self.gamma_xca * self.xca(self.norm_xca(x)))
        x = x.reshape(B, H, W, C)

        x = self.norm(x)
        x = self.mlp(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        x = shortcut + self.drop_path(x)
        return x

    SplitTransposeBlock.split_forward = split_forward
    SplitTransposeBlock.forward = split_forward


def install_edgenext_patches():
    install_xca_patch()
    install_sdta_patch()


def _strip_prefix(sd):
    for p in ("module.", "model."):
        if all(k.startswith(p) for k in sd):
            sd = {k[len(p):]: v for k, v in sd.items()}
    return sd


_HEAD_DIM_STAGE1 = {"edgenext_x_small": 16, "edgenext_small": 12, "edgenext_base": 20}


def _infer_arch(sd, dims, depths):
    arch = ARCH_BY_STAGE4_DIM.get(dims[3])
    if arch is not None:
        return arch
    if tuple(depths) == (2, 2, 6, 2):
        return "edgenext_xx_small"
    p = "stages.1.blocks.%d.xca." % (depths[1] - 1)
    if (p + "qkv.weight") in sd:
        hd = int(sd[p + "qkv.weight"].shape[0]) // (3 * int(sd[p + "temperature"].shape[0]))
        for arch, h in _HEAD_DIM_STAGE1.items():
            if h == hd:
                return arch
    raise ValueError("cannot infer the EdgeNeXt variant from this state_dict; pass --arch")


def rebuild_from_state_dict(sd, arch=None):
    """Build a timm EdgeNeXt whose every layer matches the shapes in `sd`
    (pruned stage widths, per-block MLP widths, pruned heads, pruned SDTA
    split widths), then load it strictly."""
    import timm
    import torch
    import torch.nn as nn
    install_edgenext_patches()
    sd = _strip_prefix(dict(sd))
    if "head.fc.weight" not in sd or "stem.0.weight" not in sd:
        raise ValueError("state_dict does not look like a timm EdgeNeXt (missing stem.0 / head.fc)")
    num_classes = int(sd["head.fc.weight"].shape[0])
    dims, depths = [], []
    for s in range(4):
        w = sd["stem.0.weight"] if s == 0 else sd["stages.%d.downsample.1.weight" % s]
        dims.append(int(w.shape[0]))
        b = 0
        while ("stages.%d.blocks.%d.mlp.fc1.weight" % (s, b)) in sd:
            b += 1
        depths.append(b)
    if arch is None:
        arch = _infer_arch(sd, dims, depths)
    model = timm.create_model(arch, pretrained=False, num_classes=num_classes, dims=tuple(dims))
    for s, stage in enumerate(model.stages):
        if len(stage.blocks) != depths[s]:
            raise ValueError("block count mismatch in stage %d (%d vs %d): wrong --arch?"
                             % (s, len(stage.blocks), depths[s]))
        for b, blk in enumerate(stage.blocks):
            p = "stages.%d.blocks.%d." % (s, b)
            w1, w2 = sd[p + "mlp.fc1.weight"], sd[p + "mlp.fc2.weight"]
            blk.mlp.fc1 = nn.Linear(int(w1.shape[1]), int(w1.shape[0]), bias=(p + "mlp.fc1.bias") in sd)
            blk.mlp.fc2 = nn.Linear(int(w2.shape[1]), int(w2.shape[0]), bias=(p + "mlp.fc2.bias") in sd)
            if (p + "xca.qkv.weight") in sd:
                q, pr = sd[p + "xca.qkv.weight"], sd[p + "xca.proj.weight"]
                heads = int(sd[p + "xca.temperature"].shape[0])
                blk.xca.qkv = nn.Linear(int(q.shape[1]), int(q.shape[0]), bias=(p + "xca.qkv.bias") in sd)
                blk.xca.proj = nn.Linear(int(pr.shape[1]), int(pr.shape[0]), bias=(p + "xca.proj.bias") in sd)
                blk.xca.num_heads = heads
                blk.xca.temperature = nn.Parameter(torch.ones(heads, 1, 1))
                convs = []
                for j in range(len(blk.convs)):
                    cw = sd[p + "convs.%d.weight" % j]
                    width = int(cw.shape[0])
                    convs.append(nn.Conv2d(width, width, kernel_size=int(cw.shape[-1]),
                                           padding=int(cw.shape[-1]) // 2, groups=width,
                                           bias=(p + "convs.%d.bias" % j) in sd))
                blk.convs = nn.ModuleList(convs)
                blk.width = int(convs[0].in_channels)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise ValueError("rebuild mismatch: missing=%s unexpected=%s" % (missing[:8], unexpected[:8]))
    return model.eval(), arch


def load_torch_model(path, arch=None):
    """Accepts: a pickled timm EdgeNeXt (your pruning scripts' output), a plain
    state_dict, or this script's own {format, arch, state_dict} checkpoint.
    Always returns a model rebuilt from its state_dict, so every model goes
    through the same code path."""
    import torch
    install_edgenext_patches()
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, torch.nn.Module):
        sd = obj.state_dict()
        arch = arch or None
    elif isinstance(obj, dict) and "state_dict" in obj:
        sd = obj["state_dict"]
        arch = arch or obj.get("arch")
    elif isinstance(obj, dict):
        sd = obj
    else:
        raise ValueError("unrecognised checkpoint type %r in %s" % (type(obj), path))
    model, arch = rebuild_from_state_dict(sd, arch)
    if isinstance(obj, torch.nn.Module):
        obj.eval()
        x = torch.randn(2, 3, 224, 224, generator=torch.Generator().manual_seed(0))
        with torch.inference_mode():
            d = (obj(x) - model(x)).abs().max().item()
        if d > 1e-5:
            raise ValueError("rebuilt model differs from pickled model (max abs diff %g)" % d)
    return model, arch


def count_params(model):
    return int(sum(p.numel() for p in model.parameters()))


def count_macs(model, input_size=224):
    """MACs of one 1x3xHxW forward pass: Conv2d + Linear (all, incl. positional
    encoding projection and head) + the two attention matmuls of every XCA
    (q@k^T and attn@v). Element-wise ops, norms and softmax are not counted.
    FLOPs are reported as 2 x MACs. Same definition for pruned and unpruned."""
    import torch
    import torch.nn as nn
    from timm.models.edgenext import CrossCovarianceAttn
    total = {"conv": 0, "linear": 0, "attn_matmul": 0}
    hooks = []

    def conv_hook(m, inp, out):
        total["conv"] += out.numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]

    def lin_hook(m, inp, out):
        total["linear"] += out.numel() * m.in_features

    def xca_hook(m, inp, out):
        B, N, _ = inp[0].shape
        hd = m.qkv.out_features // 3 // m.num_heads
        total["attn_matmul"] += 2 * B * m.num_heads * hd * hd * N

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(lin_hook))
        elif isinstance(m, CrossCovarianceAttn):
            hooks.append(m.register_forward_hook(xca_hook))
    with torch.inference_mode():
        model(torch.zeros(1, 3, input_size, input_size))
    for h in hooks:
        h.remove()
    macs = sum(total.values())
    return {"macs": int(macs), "flops_2x_macs": int(2 * macs), "breakdown": total,
            "definition": "Conv2d + Linear + XCA attention matmuls, batch 1; FLOPs = 2 x MACs"}


def arch_summary(model):
    """Per stage: trunk width, MLP hidden width per block, heads and the SDTA
    split [conv width x n_convs, pass-through width] of each SDTA block."""
    out = {"stage_dims": [], "mlp_hidden": [], "heads": [], "sdta_split": []}
    for s, stage in enumerate(model.stages):
        out["stage_dims"].append(int(stage.blocks[0].mlp.fc2.out_features))
        out["mlp_hidden"].append([int(b.mlp.fc1.out_features) for b in stage.blocks])
        out["heads"].append([int(b.xca.num_heads) for b in stage.blocks if hasattr(b, "xca")])
        split = []
        for b in stage.blocks:
            if hasattr(b, "convs"):
                w, n = int(b.convs[0].in_channels), len(b.convs)
                split.append([w] * n + [int(b.norm_xca.normalized_shape[0]) - w * n])
        out["sdta_split"].append(split)
    return out


def save_portable_checkpoint(model, arch, path):
    import torch
    torch.save({"format": SCHEMA, "arch": arch, "num_classes": int(model.head.fc.out_features),
                "state_dict": model.state_dict()}, path)



class TorchRunner:
    name = "pytorch"

    def __init__(self, path, threads, arch=None):
        import torch
        torch.set_num_threads(threads)
        self.torch = torch
        self.model, _ = load_torch_model(path, arch)
        self.max_batch = 32
        self.version = torch.__version__
        self.io = {"layout": "NCHW", "input_dtype": "float32"}

    def prepare(self, x):
        return self.torch.from_numpy(np.ascontiguousarray(x))

    def run(self, xt):
        with self.torch.inference_mode():
            return self.model(xt)

    def logits(self, out):
        return out.detach().cpu().numpy().astype(np.float32)


class OrtRunner:
    name = "onnx"

    def __init__(self, path, threads):
        import onnxruntime as ort
        so = ort.SessionOptions()
        so.intra_op_num_threads = threads
        so.inter_op_num_threads = 1
        so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.sess = ort.InferenceSession(path, sess_options=so, providers=["CPUExecutionProvider"])
        i = self.sess.get_inputs()[0]
        self.in_name = i.name
        self.out_name = self.sess.get_outputs()[0].name
        self.max_batch = 32 if not isinstance(i.shape[0], int) else 1
        self.version = ort.__version__
        self.io = {"layout": "NCHW", "input_dtype": "float32", "provider": "CPUExecutionProvider",
                   "input_shape": [str(d) for d in i.shape]}

    def prepare(self, x):
        return np.ascontiguousarray(x, np.float32)

    def run(self, x):
        return self.sess.run([self.out_name], {self.in_name: x})[0]

    def logits(self, out):
        return np.asarray(out, np.float32)


def _tflite_interpreter_cls():
    for mod, label in (("ai_edge_litert.interpreter", "ai-edge-litert"),
                       ("tflite_runtime.interpreter", "tflite-runtime"),
                       ("tensorflow.lite", "tensorflow")):
        try:
            m = __import__(mod, fromlist=["Interpreter"])
            return m.Interpreter, label
        except Exception:
            continue
    raise ImportError("no TFLite interpreter (install ai-edge-litert or tflite-runtime)")


class TfliteRunner:
    name = "tflite"

    def __init__(self, path, threads):
        Interpreter, label = _tflite_interpreter_cls()
        self.it = Interpreter(model_path=path, num_threads=threads)
        self.it.allocate_tensors()
        i = self.it.get_input_details()[0]
        o = self.it.get_output_details()[0]
        self.in_idx, self.out_idx = i["index"], o["index"]
        shape = list(i["shape"])
        self.nhwc = len(shape) == 4 and shape[-1] == 3 and shape[1] != 3
        self.in_dtype, self.out_dtype = i["dtype"], o["dtype"]
        self.in_q, self.out_q = i["quantization"], o["quantization"]
        self.max_batch = 1
        self.version = "%s %s" % (label, pkg_version(label) or "")
        self.io = {"layout": "NHWC" if self.nhwc else "NCHW", "input_dtype": np.dtype(self.in_dtype).name,
                   "output_dtype": np.dtype(self.out_dtype).name, "input_quant": list(map(float, self.in_q)),
                   "output_quant": list(map(float, self.out_q)), "input_shape": [int(d) for d in shape]}

    def prepare(self, x):
        if self.nhwc:
            x = x.transpose(0, 2, 3, 1)
        if np.issubdtype(self.in_dtype, np.integer):
            scale, zp = self.in_q
            if not scale:
                raise RuntimeError("quantized input without scale")
            info = np.iinfo(self.in_dtype)
            x = np.clip(np.round(x / scale + zp), info.min, info.max)
        return np.ascontiguousarray(x.astype(self.in_dtype))

    def run(self, x):
        self.it.set_tensor(self.in_idx, x)
        self.it.invoke()
        return self.it.get_tensor(self.out_idx)

    def logits(self, out):
        out = np.asarray(out)
        if np.issubdtype(self.out_dtype, np.integer):
            scale, zp = self.out_q
            return ((out.astype(np.float32) - zp) * scale).astype(np.float32)
        return out.astype(np.float32)


def make_runner(runtime, path, threads, arch=None):
    if runtime == "pytorch":
        return TorchRunner(path, threads, arch)
    if runtime == "onnx":
        return OrtRunner(path, threads)
    if runtime in ("tflite_fp32", "tflite_int8"):
        return TfliteRunner(path, threads)
    raise ValueError(runtime)


def run_logits(runner, x_batch):
    outs = []
    for i in range(0, x_batch.shape[0], runner.max_batch):
        xb = runner.prepare(x_batch[i:i + runner.max_batch])
        outs.append(runner.logits(runner.run(xb)))
    return np.concatenate(outs, 0)


def evaluate_accuracy(runner, images_u8, labels, cfg, chunk=64, progress_every=1000):
    """Returns (top-1 predictions, top-1 correct, top-5 correct) per image."""
    preds = np.empty(labels.size, np.int64)
    top5 = np.zeros(labels.size, bool)
    t0 = time.perf_counter()
    for s in range(0, labels.size, chunk):
        x = preprocess_batch(images_u8[s:s + chunk], cfg)
        lg = run_logits(runner, x)
        n = x.shape[0]
        preds[s:s + n] = lg.argmax(1)
        k = min(5, lg.shape[1])
        top = np.argpartition(-lg, k - 1, axis=1)[:, :k]
        top5[s:s + n] = (top == labels[s:s + n, None]).any(1)
        done = s + n
        if progress_every and (done // progress_every) != ((done - n) // progress_every):
            log("      %d/%d images (%.0f s)" % (done, labels.size, time.perf_counter() - t0))
    return preds, preds == labels, top5


def measure_sustained(runners, x_single, seconds, meter, idle_seconds):
    """Continuous batch-1 inference for `seconds` per model, in the order
    A B B A (half the time each) so heating affects both equally, after an idle
    window. Every window records unix start/end times (to match an external
    power meter's log), inference count, temperatures and, if the device has an
    on-board sensor, energy."""
    prepared = {k: r.prepare(x_single) for k, r in runners.items()}
    st = device_state()
    tok = meter.start()
    t0 = time.time()
    time.sleep(idle_seconds)
    idle = {"start_unix": t0, "end_unix": time.time(), "seconds": idle_seconds, "energy": meter.stop(tok),
            "temp_start_c": st["max_temp_c"], "temp_end_c": device_state()["max_temp_c"]}
    keys = list(runners)
    windows = []
    for k in keys + keys[::-1]:
        r, xp = runners[k], prepared[k]
        st0 = device_state()
        tok = meter.start()
        t_unix, tp, n = time.time(), time.perf_counter(), 0
        while time.perf_counter() - tp < seconds / 2.0:
            r.run(xp)
            n += 1
        el = time.perf_counter() - tp
        e = meter.stop(tok)
        st1 = device_state()
        windows.append({"model": k, "start_unix": t_unix, "end_unix": time.time(), "seconds": el, "inferences": n,
                        "energy": e, "temp_start_c": st0["max_temp_c"], "temp_end_c": st1["max_temp_c"],
                        "cpu_mhz_end": st1["cpu_mhz"], "throttled_end": st1["throttled"]})
    per = {}
    idle_w = idle["energy"]["avg_w"] if idle["energy"] else None
    for k in keys:
        ws = [w for w in windows if w["model"] == k]
        n, sec = sum(w["inferences"] for w in ws), sum(w["seconds"] for w in ws)
        d = {"inferences": n, "seconds": sec, "throughput_img_s": n / sec if sec else None,
             "mean_latency_ms": 1000.0 * sec / n if n else None}
        if all(w["energy"] for w in ws):
            j = sum(w["energy"]["joules"] for w in ws)
            d.update(energy_j=j, avg_power_w=j / sec, energy_mj_per_inference=1000.0 * j / n)
            if idle_w is not None:
                d["energy_mj_per_inference_above_idle"] = 1000.0 * (j - idle_w * sec) / n
        per[k] = d
    return {"idle": idle, "windows": windows, "per_model": per, "meter": meter.kind,
            "meter_detail": meter.detail}


def run_probe(runtime, path, threads, arch, x_path, steady=10, timeout=900):
    """Load and run one model in a fresh process: model load time, first
    (cold-start) inference time and memory, without other models in memory."""
    cmd = [sys.executable, os.path.abspath(__file__), "probe", "--runtime", runtime, "--model-file", path,
           "--threads", str(threads), "--input", x_path, "--steady", str(steady)]
    if arch:
        cmd += ["--arch", arch]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    for line in r.stdout.splitlines():
        if line.startswith("PROBE_JSON "):
            return json.loads(line[len("PROBE_JSON "):])
    raise RuntimeError("probe failed: %s" % (r.stderr.strip().splitlines()[-1:] or ["no output"])[0])


def cmd_probe(a):
    rss0 = rss_bytes()
    t0 = time.perf_counter()
    if a.runtime == "pytorch":
        import torch
        import timm
    elif a.runtime == "onnx":
        import onnxruntime
    else:
        _tflite_interpreter_cls()
    import_ms = 1000 * (time.perf_counter() - t0)
    rss_import = rss_bytes()
    x = np.load(a.input)
    t0 = time.perf_counter()
    r = make_runner(a.runtime, a.model_file, a.threads, a.arch)
    load_ms = 1000 * (time.perf_counter() - t0)
    rss_load = rss_bytes()
    xp = r.prepare(x)
    t0 = time.perf_counter()
    r.run(xp)
    first_ms = 1000 * (time.perf_counter() - t0)
    steady = []
    for _ in range(a.steady):
        t0 = time.perf_counter()
        r.run(xp)
        steady.append(1000 * (time.perf_counter() - t0))
    out = {"rss_start_bytes": rss0, "runtime_import_ms": import_ms, "rss_after_import_bytes": rss_import,
           "load_ms": load_ms, "rss_after_load_bytes": rss_load, "first_inference_ms": first_ms,
           "steady_median_ms": float(np.median(steady)) if steady else None,
           "rss_end_bytes": rss_bytes(), "peak_rss_bytes": peak_rss_bytes()}
    if out["peak_rss_bytes"] is not None and rss_import is not None:
        out["peak_above_runtime_bytes"] = out["peak_rss_bytes"] - rss_import
    print("PROBE_JSON " + json.dumps(out), flush=True)


def measure_latency_pair(runners, x_single, warmup, runs, rounds):
    """Batch-1 latency for several models of ONE runtime, interleaved in rounds
    (order rotated each round) so thermal throttling and background load hit all
    models equally. Times only the inference call (set_tensor+invoke+get_tensor
    for TFLite, session.run for ONNX Runtime, model(x) for PyTorch)."""
    prepared = {k: r.prepare(x_single) for k, r in runners.items()}
    for k, r in runners.items():
        for _ in range(warmup):
            r.run(prepared[k])
    lat = {k: [] for k in runners}
    per_round = max(1, runs // rounds)
    keys = list(runners)
    for rd in range(rounds):
        order = keys[rd % len(keys):] + keys[:rd % len(keys)]
        for k in order:
            r, xp = runners[k], prepared[k]
            for _ in range(per_round):
                t0 = time.perf_counter_ns()
                r.run(xp)
                lat[k].append((time.perf_counter_ns() - t0) / 1e6)
    return lat



def export_onnx(model, path, input_size, opset):
    import torch
    x = torch.zeros(1, 3, input_size, input_size)
    kwargs = dict(export_params=True, opset_version=opset, do_constant_folding=True,
                  input_names=["input"], output_names=["output"],
                  dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}})
    try:
        torch.onnx.export(model, x, path, dynamo=False, **kwargs)
    except TypeError:
        torch.onnx.export(model, x, path, **kwargs)
    import onnx
    m = onnx.load(path)
    onnx.checker.check_model(m)
    if any(t.data_location == onnx.TensorProto.EXTERNAL for t in m.graph.initializer):
        raise RuntimeError("ONNX export wrote external data; keep models self-contained")


def export_tflite_fp32(model, path, input_size):
    import torch
    try:
        import litert_torch as conv
    except ImportError:
        import ai_edge_torch as conv
    sample = (torch.zeros(1, 3, input_size, input_size),)
    edge = conv.convert(model.eval(), sample)
    edge.export(path)


def _rank_fix(mode):
    """Replacement for ai_edge_quantizer's fix_quantization_params_rank.

    The quantizer sometimes reports quantized_dimension=0 for per-channel
    constants shaped [1, 1, C] / [1, 1, 1, C] (EdgeNeXt's LayerScale gamma).
    mode="legacy" is the shim from your step2_convert_int8.py: it puts the C
    scales on axis 0, the [1,..,C] tensor broadcasts against [C,..,1] and the
    stored constant becomes C x C values, of which TFLite reads only the first
    row (wrong for every channel but one). mode="axis_match" (default) puts the
    scales on the axis whose size equals the number of scales."""
    import dataclasses

    def fix(tensor_data, qp):
        scales, zps = np.asarray(qp.scale), np.asarray(qp.zero_point)
        if mode == "legacy":
            if tensor_data.ndim == scales.ndim:
                return qp
            if tensor_data.ndim == 0:
                scales, zps = np.array(scales.item()), np.array(zps.item())
            else:
                scales, zps = np.squeeze(scales), np.squeeze(zps)
                if scales.ndim == 0:
                    scales = np.reshape(scales, [1] * tensor_data.ndim)
                    zps = np.reshape(zps, [1] * tensor_data.ndim)
                else:
                    shape = [1] * tensor_data.ndim
                    shape[qp.quantized_dimension or 0] = scales.shape[0]
                    scales, zps = np.reshape(scales, shape), np.reshape(zps, shape)
            return dataclasses.replace(qp, scale=scales, zero_point=zps)
        if tensor_data.ndim == 0:
            if scales.size != 1:
                raise ValueError("scalar tensor with %d scales" % scales.size)
            return dataclasses.replace(qp, scale=np.array(scales.item()), zero_point=np.array(zps.item()))
        n = int(scales.size)
        if zps.size == 1 and n > 1:
            zps = np.full(n, zps.reshape(-1)[0], dtype=zps.dtype)
        shape = [1] * tensor_data.ndim
        qd = qp.quantized_dimension
        if n > 1:
            if qd is None or qd >= tensor_data.ndim or tensor_data.shape[qd] != n:
                match = [d for d, s in enumerate(tensor_data.shape) if s == n]
                if not match:
                    raise ValueError("cannot place %d scales on a tensor of shape %s" % (n, tensor_data.shape))
                qd = match[-1]
            shape[qd] = n
        return dataclasses.replace(qp, scale=scales.reshape(shape), zero_point=zps.reshape(shape),
                                   quantized_dimension=qd)
    return fix


def tflite_malformed_constants(path):
    """Count constant tensors whose stored bytes do not match their shape x dtype
    (the symptom of the legacy rank fix). Returns None if flatbuffer tools are missing."""
    try:
        from ai_edge_litert.tools import flatbuffer_utils as fbu
    except ImportError:
        return None
    itemsize = {0: 4, 1: 2, 2: 4, 3: 1, 4: 8, 7: 2, 9: 1}
    m = fbu.read_model(path)
    bad = 0
    for sg in m.subgraphs:
        for t in sg.tensors:
            buf = m.buffers[t.buffer].data if t.buffer < len(m.buffers) else None
            if buf is None or len(buf) == 0 or t.type not in itemsize:
                continue
            if len(buf) != itemsize[t.type] * int(np.prod(t.shape if len(t.shape) else [1])):
                bad += 1
    return bad


def quantize_tflite_int8(fp32_path, out_path, calib_x, rank_fix="axis_match"):
    """Full static INT8 with AI Edge Quantizer, positional encoding kept FP32,
    same recipe as your step2_convert_int8.py (with the corrected rank fix)."""
    from ai_edge_quantizer import quantizer, recipe_manager, qtyping, recipe
    try:
        from ai_edge_quantizer.algorithms.uniform_quantize import uniform_quantize_tensor as uqt
        if not hasattr(uqt, "_edgenext_bench_original"):
            uqt._edgenext_bench_original = uqt.fix_quantization_params_rank
        uqt.fix_quantization_params_rank = _rank_fix(rank_fix) if rank_fix else uqt._edgenext_bench_original
    except ImportError:
        pass

    Interpreter, _ = _tflite_interpreter_cls()
    sigs = Interpreter(model_path=fp32_path).get_signature_list()
    in_name = "args_0"
    if sigs:
        first = sigs[next(iter(sigs))]
        if first.get("inputs"):
            in_name = first["inputs"][0]

    rp = recipe_manager.RecipeManager()
    rp.add_static_config(regex=".*", operation_name=qtyping.TFLOperationName.ALL_SUPPORTED,
                         activation_num_bits=8, weight_num_bits=8)
    rp.add_quantization_config(regex=".*PositionalEncodingFourier.*",
                               operation_name=qtyping.TFLOperationName.ALL_SUPPORTED,
                               op_config=None, algorithm_key=recipe.AlgorithmName.NO_QUANTIZE)
    q = quantizer.Quantizer(fp32_path)
    q.load_quantization_recipe(rp.get_quantization_recipe())
    calib = {None: [{in_name: calib_x[i:i + 1]} for i in range(calib_x.shape[0])]}
    res = q.quantize(q.calibrate(calib))
    tmp = out_path + ".tmp.tflite"
    res.export_model(tmp, overwrite=True)
    try:
        from ai_edge_litert.tools import flatbuffer_utils as fbu
        m = fbu.read_model(tmp)
        for t in m.subgraphs[0].tensors:
            qn = t.quantization
            if qn is not None and qn.scale is not None and len(qn.scale) > 1:
                if t.shape[qn.quantizedDimension] != len(qn.scale):
                    match = [d for d, s in enumerate(t.shape) if s == len(qn.scale)]
                    if match:
                        qn.quantizedDimension = match[0]
        fbu.write_model(m, out_path)
        os.remove(tmp)
    except ImportError:
        os.replace(tmp, out_path)



def cmd_export(a):
    os.environ.setdefault("TQDM_DISABLE", "1")
    import torch
    torch.manual_seed(a.seed)
    out = os.path.abspath(a.out)
    os.makedirs(os.path.join(out, "models"), exist_ok=True)
    ds = DATASETS[a.dataset]
    cfg = dict(ds["preprocess"])
    cfg["size"] = a.input_size
    class_order = a.class_order.split(",") if a.class_order else None
    warnings = []

    log("[1/7] loading models")
    models = {}
    for role, path in (("pruned", a.pruned), ("unpruned", a.unpruned)):
        m, arch = load_torch_model(path, a.arch)
        models[role] = {"model": m, "arch": arch, "source": os.path.abspath(path),
                        "source_sha256": sha256_file(path)}
        log("  %s: %s (%s)" % (role, os.path.basename(path), arch))
    if models["pruned"]["arch"] != models["unpruned"]["arch"]:
        if not a.allow_arch_mismatch:
            raise SystemExit("pruned and unpruned are different architectures: %s vs %s (pass "
                             "--allow-arch-mismatch to compare a pruned model with a smaller variant)"
                             % (models["pruned"]["arch"], models["unpruned"]["arch"]))
        warnings.append("reference ('unpruned') model is %s, pruned model is %s"
                        % (models["unpruned"]["arch"], models["pruned"]["arch"]))
    if models["pruned"]["source_sha256"] == models["unpruned"]["source_sha256"]:
        raise SystemExit("pruned and unpruned are the same file")
    n_classes = dataset_num_classes(a.dataset, a.data_root)
    for role in models:
        nc = int(models[role]["model"].head.fc.out_features)
        if nc != n_classes:
            raise SystemExit("%s model has %d outputs but %s has %d classes"
                             % (role, nc, a.dataset, n_classes))

    log("[2/7] parameters and MACs")
    for role, d in models.items():
        d["params"] = count_params(d["model"])
        d["complexity"] = count_macs(d["model"], a.input_size)
        d["arch_summary"] = arch_summary(d["model"])
        log("  %s: %s params, %.2f M MACs" % (role, f"{d['params']:,}", d["complexity"]["macs"] / 1e6))
    ratio = models["pruned"]["params"] / models["unpruned"]["params"]
    if ratio > 0.99:
        warnings.append("pruned model keeps %.2f%% of the unpruned parameters - it is essentially "
                        "unpruned" % (100 * ratio))

    log("[3/7] evaluation and calibration images")
    eval_imgs, eval_labels, class_names = load_raw_split(a.dataset, a.data_root, ds["eval_split"],
                                                         class_order, a.download, per_class=a.eval_per_class)
    if a.dataset in ("tiny10", "tiny200"):
        class_order = class_names
    if a.dataset in ("cifar10", "cifar100"):
        train_imgs, train_labels, _ = load_raw_split(a.dataset, a.data_root, "train", class_order, a.download)
        calib_idx = stratified_indices(train_labels, total=a.calib_n, num_classes=n_classes, spread=True)
        calib_rule = "class-balanced, evenly spaced within each class"
    else:
        k = max(1, int(math.ceil(a.calib_n / float(n_classes))))
        train_imgs, train_labels, _ = load_raw_split(a.dataset, a.data_root, "train", class_order,
                                                     a.download, per_class=k)
        calib_idx = np.arange(train_labels.size)
        calib_rule = "first %d images (sorted file names) of every class; indices refer to that list" % k
    calib_x = preprocess_batch(train_imgs[calib_idx], cfg)
    ref_x = preprocess_batch(eval_imgs[:4], cfg)
    evalset = os.path.join(out, "evalset.npz")
    np.savez_compressed(evalset, images=eval_imgs, labels=eval_labels, ref_preprocessed=ref_x)
    log("  eval: %d %s images; calibration: %d train images" % (eval_labels.size, ds["eval_split"], calib_idx.size))

    check_x = preprocess_batch(eval_imgs[:16], cfg)
    xt = torch.from_numpy(check_x)
    for role, d in models.items():
        with torch.inference_mode():
            d["ref_logits"] = d["model"](xt).numpy()

    log("[4/7] PyTorch checkpoints")
    for role, d in models.items():
        p = os.path.join(out, "models", "%s.pt" % role)
        save_portable_checkpoint(d["model"], d["arch"], p)
        d["files"] = {"pytorch": file_record(p, out)}

    log("[5/7] ONNX")
    for role, d in models.items():
        p = os.path.join(out, "models", "%s.onnx" % role)
        export_onnx(d["model"], p, a.input_size, a.opset)
        d["files"]["onnx"] = file_record(p, out)
        r = OrtRunner(p, threads=1)
        d.setdefault("export_checks", {})["onnx_max_abs_diff_vs_torch"] = float(
            np.abs(run_logits(r, check_x) - d["ref_logits"]).max())

    if not a.skip_tflite:
        log("[6/7] TFLite FP32")
        for role, d in models.items():
            p = os.path.join(out, "models", "%s_fp32.tflite" % role)
            export_tflite_fp32(d["model"], p, a.input_size)
            d["files"]["tflite_fp32"] = file_record(p, out)
            r = TfliteRunner(p, threads=1)
            d["export_checks"]["tflite_fp32_max_abs_diff_vs_torch"] = float(
                np.abs(run_logits(r, check_x) - d["ref_logits"]).max())
            d.setdefault("tflite_memory", {})["fp32"] = tflite_memory_estimate(p)
        if not a.skip_int8:
            log("[7/7] TFLite INT8 (full integer, %d calibration images)" % calib_idx.size)
            for role, d in models.items():
                fp32 = os.path.join(out, d["files"]["tflite_fp32"]["file"])
                p = os.path.join(out, "models", "%s_int8.tflite" % role)
                quantize_tflite_int8(fp32, p, calib_x)
                d["files"]["tflite_int8"] = file_record(p, out)
                r = TfliteRunner(p, threads=1)
                agree = run_logits(r, check_x).argmax(1) == d["ref_logits"].argmax(1)
                d["export_checks"]["tflite_int8_top1_agreement_vs_torch_on_16"] = float(agree.mean())
                bad = tflite_malformed_constants(p)
                d["export_checks"]["tflite_int8_malformed_constants"] = bad
                d.setdefault("tflite_memory", {})["int8"] = tflite_memory_estimate(p)
                if bad:
                    warnings.append("%s INT8 model has %d malformed constant tensors" % (role, bad))

    for role, d in models.items():
        for k, v in d.get("export_checks", {}).items():
            if "max_abs_diff" in k and v > 1e-3:
                warnings.append("%s %s = %.2e (export not numerically faithful)" % (role, k, v))

    script_copy = os.path.join(out, "edgenext_bench.py")
    if os.path.abspath(__file__) != script_copy:
        shutil.copy2(os.path.abspath(__file__), script_copy)

    manifest = {
        "schema": SCHEMA, "created": now_iso(), "experiment": a.experiment,
        "host_env": env_info(),
        "script_sha256": sha256_file(script_copy),
        "dataset": {"name": a.dataset, "eval_split": ds["eval_split"], "num_classes": n_classes,
                    "class_names": class_names, "n_eval": int(eval_labels.size),
                    "eval_per_class_cap": a.eval_per_class,
                    "preprocess": cfg, "evalset": file_record(evalset, out)},
        "calibration": {"split": "train", "n": int(calib_idx.size), "indices": calib_idx.tolist(),
                        "rule": calib_rule},
        "models": {role: {k: v for k, v in d.items() if k not in ("model", "ref_logits")}
                   for role, d in models.items()},
        "warnings": warnings,
    }
    write_json(os.path.join(out, "manifest.json"), manifest)
    log("\nwrote %s" % os.path.join(out, "manifest.json"))
    for w in warnings:
        log("WARNING: " + w)



def cmd_bench(a):
    import gc
    import tempfile
    art = os.path.abspath(a.artifacts)
    man_path = os.path.join(art, "manifest.json")
    man = read_json(man_path)
    if man.get("schema") != SCHEMA:
        raise SystemExit("unknown manifest schema")
    threads = a.threads or min(4, os.cpu_count() or 1)

    log("verifying checksums")
    ev = man["dataset"]["evalset"]
    if sha256_file(os.path.join(art, ev["file"])) != ev["sha256"]:
        raise SystemExit("evalset.npz does not match the manifest")
    for role, m in man["models"].items():
        for rt, rec in m["files"].items():
            p = os.path.join(art, rec["file"])
            if os.path.exists(p) and sha256_file(p) != rec["sha256"]:
                raise SystemExit("%s %s file %s does not match the manifest" % (role, rt, rec["file"]))

    data = np.load(os.path.join(art, ev["file"]))
    images, labels = data["images"], data["labels"]
    cfg = man["dataset"]["preprocess"]
    nc = man["dataset"]["num_classes"]
    idx = stratified_indices(labels, per_class=a.per_class, total=a.limit, num_classes=nc)
    subset = idx.size < labels.size
    images, labels = images[idx], labels[idx]
    pre_diff = float(np.abs(preprocess_batch(data["images"][:4], cfg) - data["ref_preprocessed"]).max())
    notes = []
    if sha256_file(os.path.abspath(__file__)) != man.get("script_sha256"):
        notes.append("this script differs from the copy saved at export (use the one inside the "
                     "artifacts folder)")
    if pre_diff > 1e-4:
        notes.append("preprocessing on this device differs from the export host (max abs %.2e); "
                     "check the Pillow version" % pre_diff)
    log("evaluating on %d of %d %s images%s" % (labels.size, data["labels"].size,
                                                 man["dataset"]["eval_split"], " (SUBSET)" if subset else ""))

    hints = detect_emulation()
    emulated = {"yes": True, "no": False}.get(a.emulated, looks_emulated(hints))
    wanted = a.runtimes.split(",") if a.runtimes else ALL_RUNTIMES
    x_single = preprocess(images[0], cfg)[None]
    meter = PowerMeter()
    if a.sustained_seconds > 0:
        log("power sensor: %s" % (meter.detail or meter.reason))
    state_start = device_state()
    runs, runtime_info = [], []
    tmpdir = tempfile.mkdtemp(prefix="bench_probe_")
    x_path = os.path.join(tmpdir, "x.npy")
    np.save(x_path, x_single)

    for rt in wanted:
        log("\n== %s ==" % rt)
        info = {"runtime": rt, "state_before": device_state()}
        runners, recs = {}, {}
        for role, m in man["models"].items():
            rec = {"runtime": rt, "model": role}
            f = m["files"].get(rt)
            if f is None or not os.path.exists(os.path.join(art, f["file"])):
                rec.update(status="skipped", reason="no %s artifact in this package" % rt)
            else:
                rec.update(file=f["file"], file_sha256=f["sha256"], file_bytes=f["bytes"])
                try:
                    runners[role] = make_runner(rt, os.path.join(art, f["file"]), threads, m.get("arch"))
                    rec.update(status="ok", runtime_version=runners[role].version, io=runners[role].io)
                except Exception as e:
                    rec.update(status="skipped", reason="%s: %s" % (type(e).__name__, e))
            recs[role] = rec
        try:
            for role, r in runners.items():
                out_dim = run_logits(r, x_single).shape[1]
                if out_dim != nc:
                    raise RuntimeError("%s outputs %d classes, manifest says %d" % (role, out_dim, nc))
                log("  accuracy: %s" % role)
                preds, correct, c5 = evaluate_accuracy(r, images, labels, cfg, progress_every=a.progress)
                k, k5, n = int(correct.sum()), int(c5.sum()), int(correct.size)
                recs[role]["accuracy"] = {"correct": k, "n": n, "top1": k / n, "wilson95": wilson_interval(k, n),
                                          "top5_correct": k5, "top5": k5 / n, "top5_wilson95": wilson_interval(k5, n)}
                recs[role]["predictions"] = preds.tolist()
                log("    top-1 %d/%d = %.2f%%   top-5 %.2f%%" % (k, n, 100.0 * k / n, 100.0 * k5 / n))
            if runners:
                log("  latency: %d warm-up + %d timed runs per model, %d interleaved rounds, %d threads"
                    % (a.warmup, a.runs, a.rounds, threads))
                lat = measure_latency_pair(runners, x_single, a.warmup, a.runs, a.rounds)
                for role, ms in lat.items():
                    recs[role]["latency_ms"] = latency_summary(ms)
                    recs[role]["latency_raw_ms"] = [round(v, 4) for v in ms]
                    log("    %s: median %.3f ms, mean %.3f ms" % (role, np.median(ms), np.mean(ms)))
            if runners and a.sustained_seconds > 0:
                log("  sustained: %.0f s per model (A B B A) after %.0f s idle" % (a.sustained_seconds,
                                                                                  a.idle_seconds))
                sus = measure_sustained(runners, x_single, a.sustained_seconds, meter, a.idle_seconds)
                info["sustained"] = {k: v for k, v in sus.items() if k != "per_model"}
                for role, d in sus["per_model"].items():
                    recs[role]["sustained"] = d
                    log("    %s: %.2f img/s%s" % (role, d["throughput_img_s"] or 0,
                                                 ", %.2f mJ/inference" % d["energy_mj_per_inference"]
                                                 if "energy_mj_per_inference" in d else ""))
        except Exception as e:
            for role in runners:
                recs[role].update(status="error", reason="%s: %s" % (type(e).__name__, e))
                recs[role].pop("accuracy", None)
                recs[role].pop("predictions", None)
            log("  ERROR: %s: %s" % (type(e).__name__, e))
        ok_roles = [r for r in runners if recs[r]["status"] == "ok"]
        del runners
        gc.collect()
        info["state_after"] = device_state()
        if a.mem_probes > 0:
            for role in ok_roles:
                m = man["models"][role]
                pr = []
                for _ in range(a.mem_probes):
                    try:
                        pr.append(run_probe(rt, os.path.join(art, m["files"][rt]["file"]), threads,
                                            m.get("arch"), x_path))
                    except Exception as e:
                        recs[role]["probe_error"] = "%s: %s" % (type(e).__name__, e)
                        break
                if pr:
                    med = {k: float(np.median([p[k] for p in pr if p.get(k) is not None]))
                           for k in pr[0] if any(p.get(k) is not None for p in pr)}
                    recs[role]["probe"] = {"runs": pr, "median": med}
                    log("  %s: load %.1f ms, first inference %.1f ms, peak RSS %.1f MiB (%.1f MiB above the "
                        "runtime)" % (role, med.get("load_ms", float("nan")),
                                      med.get("first_inference_ms", float("nan")),
                                      med.get("peak_rss_bytes", float("nan")) / 2 ** 20,
                                      med.get("peak_above_runtime_bytes", float("nan")) / 2 ** 20))
        for rec in recs.values():
            if rec["status"] != "ok":
                log("  %s: skipped (%s)" % (rec["model"], rec["reason"]))
        runs.extend(recs.values())
        runtime_info.append(info)
    shutil.rmtree(tmpdir, ignore_errors=True)

    result = {
        "schema": SCHEMA, "created": now_iso(), "device_label": a.device_label,
        "emulated": emulated, "emulation_hints": hints, "env": env_info(),
        "experiment": man["experiment"], "manifest_sha256": sha256_file(man_path),
        "script_sha256": sha256_file(os.path.abspath(__file__)),
        "models": {r: {"arch": m["arch"], "params": m["params"], "complexity": m["complexity"],
                       "files": m["files"], "source_sha256": m.get("source_sha256"),
                       "tflite_memory": m.get("tflite_memory")} for r, m in man["models"].items()},
        "num_classes": nc,
        "settings": {"threads": threads, "warmup": a.warmup, "runs": a.runs, "rounds": a.rounds,
                     "batch_size_latency": 1, "timed_section": "inference call only",
                     "sustained_seconds": a.sustained_seconds, "idle_seconds": a.idle_seconds,
                     "mem_probes": a.mem_probes},
        "power_sensor": {"kind": meter.kind, "detail": meter.detail, "reason_if_none": meter.reason},
        "device_state": {"start": state_start, "end": device_state()},
        "runtime_info": runtime_info,
        "eval": {"dataset": man["dataset"]["name"], "split": man["dataset"]["eval_split"],
                 "n_available": int(data["labels"].size), "n_used": int(labels.size), "subset": subset,
                 "indices": idx.tolist(), "labels": labels.tolist(),
                 "preprocess_max_abs_diff_vs_host": pre_diff},
        "notes": notes, "runs": runs,
    }
    out = a.out or os.path.join(art, "results", "%s__%s.json" % (
        a.device_label.replace(" ", "_"), datetime.now().strftime("%Y%m%d-%H%M%S")))
    write_json(out, result)
    log("\nwrote %s" % out)



def _fmt(v, nd=2):
    return "-" if v is None else ("%.*f" % (nd, v))


def _mib(b):
    return b / (1024.0 * 1024.0)


def read_power_csv(path):
    """External power-meter log -> (unix_times, watts). Columns (header, any
    case): a time column (unix_time | timestamp | time | t; unix seconds or ISO
    date-time in this computer's local time) and either watts (watts | power_w |
    power | w) or volts + amps (volts | voltage | v, amps | current | a)."""
    rows = list(csv.DictReader(open(path, newline="", encoding="utf-8-sig")))
    if not rows:
        raise ValueError("empty power log %s" % path)
    cols = {c.strip().lower(): c for c in rows[0]}

    def pick(names):
        return next((cols[n] for n in names if n in cols), None)
    tc = pick(["unix_time", "timestamp", "time", "t"])
    wc = pick(["watts", "power_w", "power", "w"])
    vc, ac = pick(["volts", "voltage", "v"]), pick(["amps", "current", "a"])
    if tc is None or (wc is None and (vc is None or ac is None)):
        raise ValueError("%s needs a time column and watts (or volts + amps)" % path)
    ts, ws = [], []
    for r in rows:
        t = r[tc].strip()
        try:
            tv = float(t)
        except ValueError:
            tv = datetime.fromisoformat(t).timestamp()
        ts.append(tv)
        ws.append(float(r[wc]) if wc else float(r[vc]) * float(r[ac]))
    o = np.argsort(ts)
    return np.asarray(ts)[o], np.asarray(ws)[o]


def _meter_window(ts, ws, start, end):
    sel = (ts >= start) & (ts <= end)
    if sel.sum() < 2:
        return None
    w = float(ws[sel].mean())
    return {"avg_w": w, "joules": w * (end - start), "samples": int(sel.sum())}


def sustained_energy(r, rt, power_logs):
    """Per-model sustained metrics of one runtime in one results file, with
    energy from the on-board sensor or, if given, the external meter log."""
    info = next((x for x in r.get("runtime_info", []) if x["runtime"] == rt and x.get("sustained")), None)
    if info is None:
        return None
    sus = info["sustained"]
    log_ = power_logs.get(r["device_label"])
    src = "external meter" if log_ is not None else (sus.get("meter") or None)
    idle = sus["idle"]
    idle_e = _meter_window(log_[0], log_[1], idle["start_unix"], idle["end_unix"]) if log_ is not None \
        else idle.get("energy")
    out = {"source": src, "idle_w": idle_e["avg_w"] if idle_e else None, "models": {}}
    temps = [w.get("temp_start_c") for w in sus["windows"]] + [w.get("temp_end_c") for w in sus["windows"]]
    temps = [t for t in temps if t is not None]
    out["temp_min_c"], out["temp_max_c"] = (min(temps), max(temps)) if temps else (None, None)
    for role in {w["model"] for w in sus["windows"]}:
        ws = [w for w in sus["windows"] if w["model"] == role]
        n, sec = sum(w["inferences"] for w in ws), sum(w["seconds"] for w in ws)
        es = [(_meter_window(log_[0], log_[1], w["start_unix"], w["end_unix"]) if log_ is not None
               else w.get("energy")) for w in ws]
        d = {"throughput_img_s": n / sec if sec else None, "inferences": n, "seconds": sec}
        if all(es) and n:
            j = sum(e["joules"] for e in es)
            d.update(avg_power_w=j / sec, mj_per_inference=1000.0 * j / n)
            if out["idle_w"] is not None:
                d["mj_per_inference_above_idle"] = 1000.0 * (j - out["idle_w"] * sec) / n
        out["models"][role] = d
    return out


def _pair(u, p, fmt):
    return "%s / %s" % (fmt % u if u is not None else "-", fmt % p if p is not None else "-")


def cmd_report(a):
    files = []
    for pat in a.results:
        files += glob.glob(pat)
    files = sorted(set(files))
    if not files:
        raise SystemExit("no results files")
    results = [read_json(f) for f in files]
    for r, f in zip(results, files):
        r["_file"] = os.path.basename(f)
    exps = sorted({(r["experiment"], r["manifest_sha256"]) for r in results})
    os.makedirs(a.out, exist_ok=True)
    md, rows_csv, warn = [], [], []
    power_logs = {}
    for spec in a.power_csv or []:
        lbl, path = spec.split("=", 1)
        power_logs[lbl] = read_power_csv(path)

    for exp, msha in exps:
        rs = [r for r in results if r["manifest_sha256"] == msha]
        m0 = rs[0]["models"]
        md.append("# %s\n" % exp)
        md.append("Manifest SHA-256 `%s`. All values below are computed from the results files "
                  "listed at the end; none are entered by hand.\n" % msha[:16])

        md.append("## Model complexity (computed at export)\n")
        md.append("| | Unpruned | Pruned | Reduction |")
        md.append("|---|---|---|---|")
        pu, pp = m0["unpruned"]["params"], m0["pruned"]["params"]
        mu, mp = m0["unpruned"]["complexity"]["macs"], m0["pruned"]["complexity"]["macs"]
        md.append("| Parameters | %s | %s | %.2f%% |" % (f"{pu:,}", f"{pp:,}", 100 * (1 - pp / pu)))
        md.append("| MACs (M) | %.2f | %.2f | %.2f%% |" % (mu / 1e6, mp / 1e6, 100 * (1 - mp / mu)))
        for rt in ALL_RUNTIMES:
            fu, fp = m0["unpruned"]["files"].get(rt), m0["pruned"]["files"].get(rt)
            if fu and fp:
                md.append("| %s file (bytes / MiB) | %s / %.2f | %s / %.2f | %.2f%% |" % (
                    rt, f"{fu['bytes']:,}", _mib(fu["bytes"]), f"{fp['bytes']:,}", _mib(fp["bytes"]),
                    100 * (1 - fp["bytes"] / fu["bytes"])))
        for kind in ("fp32", "int8"):
            tu = (m0["unpruned"].get("tflite_memory") or {}).get(kind)
            tp = (m0["pruned"].get("tflite_memory") or {}).get(kind)
            if tu and tp:
                for key, label in (("weights_bytes", "weights"), ("peak_activation_bytes", "peak activation memory")):
                    md.append("| TFLite %s %s (KiB, estimate) | %.1f | %.1f | %.2f%% |" % (
                        kind.upper(), label, tu[key] / 1024, tp[key] / 1024, 100 * (1 - tp[key] / tu[key])))
        md.append("\nMACs: %s. TFLite memory: %s; the microcontroller arena is measured by its firmware.\n"
                  % (m0["unpruned"]["complexity"]["definition"],
                     "liveness estimate of activation tensors over the op order"))
        if pp / pu > 0.99:
            warn.append("%s: pruned model keeps %.2f%% of parameters (essentially unpruned)" % (exp, 100 * pp / pu))

        md.append("## Pruned vs unpruned on each device and runtime\n")
        md.append("| Device | Runtime | Images | Unpruned top-1 | Pruned top-1 | Δ (pp) | McNemar p | "
                  "Unpruned median ms | Pruned median ms | Speedup (95% CI) | Threads |")
        md.append("|---|---|---|---|---|---|---|---|---|---|---|")
        fig_rows = []
        for r in rs:
            dev = r["device_label"] + (" (emulated)" if r["emulated"] else "")
            if r["emulated"]:
                warn.append("%s: %s is emulated - latency is not hardware performance" % (exp, r["device_label"]))
            if r["eval"]["subset"]:
                warn.append("%s: %s used %d of %d images" % (exp, r["device_label"], r["eval"]["n_used"],
                                                             r["eval"]["n_available"]))
            for n in r.get("notes", []):
                warn.append("%s: %s: %s" % (exp, r["device_label"], n))
            states = [r.get("device_state", {}).get("start"), r.get("device_state", {}).get("end")]
            states += [x.get(k) for x in r.get("runtime_info", []) for k in ("state_before", "state_after")]
            states = [x for x in states if x]
            hot = [x["max_temp_c"] for x in states if x.get("max_temp_c") is not None]
            if hot and max(hot) >= 80:
                warn.append("%s: %s reached %.1f °C - check cooling" % (exp, r["device_label"], max(hot)))
            flags = sorted({k for x in states if x.get("throttled") for k, v in x["throttled"].items()
                            if k.endswith("_now") and v})
            if flags:
                warn.append("%s: %s reported %s during the run - latency may be throttled"
                            % (exp, r["device_label"], ", ".join(flags)))
            labels = np.asarray(r["eval"]["labels"])
            for rt in ALL_RUNTIMES:
                by = {x["model"]: x for x in r["runs"] if x["runtime"] == rt}
                if not by:
                    continue
                u, p = by.get("unpruned", {}), by.get("pruned", {})
                if u.get("status") != "ok" or p.get("status") != "ok":
                    reason = "; ".join("%s: %s" % (k, v.get("reason")) for k, v in by.items()
                                       if v.get("status") != "ok")
                    md.append("| %s | %s | - | - | - | - | - | - | - | not measured (%s) | - |" % (dev, rt, reason))
                    continue
                cu = np.asarray(u["predictions"]) == labels
                cp = np.asarray(p["predictions"]) == labels
                mc = mcnemar_exact(cp, cu)
                lu, lp = u["latency_raw_ms"], p["latency_raw_ms"]
                sp = float(np.median(lu) / np.median(lp))
                ci = bootstrap_ratio_of_medians(lu, lp)
                acc_u, acc_p = u["accuracy"]["top1"], p["accuracy"]["top1"]
                md.append("| %s | %s | %d | %.2f%% | %.2f%% | %+.2f | %.3g | %.3f | %.3f | %.3fx (%.3f-%.3f) | %d |" % (
                    dev, rt, cp.size, 100 * acc_u, 100 * acc_p, 100 * (acc_p - acc_u), mc["p_value"],
                    np.median(lu), np.median(lp), sp, ci[0], ci[1], r["settings"]["threads"]))
                fig_rows.append((r["device_label"], rt, float(np.median(lu)), float(np.median(lp))))
                se = sustained_energy(r, rt, power_logs) or {"models": {}, "source": None}
                for role, x in (("unpruned", u), ("pruned", p)):
                    sm = se["models"].get(role, {})
                    L = x["latency_ms"]
                    rows_csv.append({
                        "experiment": exp, "device": r["device_label"], "emulated": r["emulated"],
                        "runtime": rt, "runtime_version": x.get("runtime_version"), "model": role,
                        "params": r["models"][role]["params"], "macs": r["models"][role]["complexity"]["macs"],
                        "file_bytes": x["file_bytes"], "images": x["accuracy"]["n"],
                        "correct": x["accuracy"]["correct"], "top1": x["accuracy"]["top1"],
                        "top1_ci_lo": x["accuracy"]["wilson95"][0], "top1_ci_hi": x["accuracy"]["wilson95"][1],
                        "lat_median_ms": L["median"], "lat_mean_ms": L["mean"], "lat_std_ms": L["std"],
                        "lat_p95_ms": L["p95"], "lat_min_ms": L["min"], "lat_max_ms": L["max"],
                        "throughput_img_s": L["throughput_img_s_from_median"], "latency_runs": L["n"],
                        "top5": x["accuracy"].get("top5"),
                        "load_ms": (x.get("probe") or {}).get("median", {}).get("load_ms"),
                        "first_inference_ms": (x.get("probe") or {}).get("median", {}).get("first_inference_ms"),
                        "peak_rss_mib": ((x.get("probe") or {}).get("median", {}).get("peak_rss_bytes") or 0)
                        / 2 ** 20 or None,
                        "sustained_img_s": sm.get("throughput_img_s"),
                        "energy_mj_per_inference": sm.get("mj_per_inference"),
                        "energy_mj_per_inference_above_idle": sm.get("mj_per_inference_above_idle"),
                        "avg_power_w": sm.get("avg_power_w"), "energy_source": se["source"],
                        "threads": r["settings"]["threads"], "results_file": r["_file"]})

        md.append("\nSpeedup = unpruned median / pruned median latency (batch 1); CI by bootstrap. "
                  "McNemar compares per-image correctness of the two models on the same images.\n")

        def ok_pairs():
            for r in rs:
                for rt in ALL_RUNTIMES:
                    by = {x["model"]: x for x in r["runs"] if x["runtime"] == rt and x.get("status") == "ok"}
                    if "unpruned" in by and "pruned" in by:
                        yield r, rt, by["unpruned"], by["pruned"]

        if (rs[0].get("num_classes") or 0) > 10:
            md.append("## Top-5 accuracy\n")
            md.append("| Device | Runtime | Images | Unpruned top-5 | Pruned top-5 | Δ (pp) |")
            md.append("|---|---|---|---|---|---|")
            for r, rt, u, p in ok_pairs():
                if "top5" in u["accuracy"]:
                    md.append("| %s | %s | %d | %.2f%% | %.2f%% | %+.2f |" % (
                        r["device_label"], rt, u["accuracy"]["n"], 100 * u["accuracy"]["top5"],
                        100 * p["accuracy"]["top5"], 100 * (p["accuracy"]["top5"] - u["accuracy"]["top5"])))
            md.append("")

        mem_rows = [(r, rt, u, p) for r, rt, u, p in ok_pairs() if u.get("probe") and p.get("probe")]
        if mem_rows:
            md.append("## Start-up time and memory (unpruned / pruned, fresh process, median of %d)\n"
                      % mem_rows[0][0]["settings"].get("mem_probes", 0))
            md.append("| Device | Runtime | Model load (ms) | First inference (ms) | Peak RAM of the process (MiB) | "
                      "RAM above the bare runtime (MiB) |")
            md.append("|---|---|---|---|---|---|")
            for r, rt, u, p in mem_rows:
                mu, mp = u["probe"]["median"], p["probe"]["median"]
                mib = lambda d, k: d[k] / 2 ** 20 if d.get(k) is not None else None
                md.append("| %s | %s | %s | %s | %s | %s |" % (
                    r["device_label"], rt, _pair(mu.get("load_ms"), mp.get("load_ms"), "%.1f"),
                    _pair(mu.get("first_inference_ms"), mp.get("first_inference_ms"), "%.1f"),
                    _pair(mib(mu, "peak_rss_bytes"), mib(mp, "peak_rss_bytes"), "%.1f"),
                    _pair(mib(mu, "peak_above_runtime_bytes"), mib(mp, "peak_above_runtime_bytes"), "%.1f")))
            md.append("\nLoad = creating the session / interpreter (file already in the OS cache); first "
                      "inference = the first call after loading (cold start).\n")

        sus_rows = []
        for r, rt, u, p in ok_pairs():
            e = sustained_energy(r, rt, power_logs)
            if e:
                sus_rows.append((r, rt, e))
        if sus_rows:
            md.append("## Sustained inference, temperature and energy (unpruned / pruned)\n")
            md.append("| Device | Runtime | Seconds per model | Throughput (img/s) | Temperature (°C) | "
                      "Energy per inference (mJ) | Above idle (mJ) | Avg power (W) | Energy source |")
            md.append("|---|---|---|---|---|---|---|---|---|")
            for r, rt, e in sus_rows:
                u, p = e["models"].get("unpruned", {}), e["models"].get("pruned", {})
                md.append("| %s | %s | %.0f | %s | %s | %s | %s | %s | %s |" % (
                    r["device_label"], rt, r["settings"].get("sustained_seconds", 0),
                    _pair(u.get("throughput_img_s"), p.get("throughput_img_s"), "%.2f"),
                    ("%.1f-%.1f" % (e["temp_min_c"], e["temp_max_c"])) if e["temp_min_c"] is not None else "-",
                    _pair(u.get("mj_per_inference"), p.get("mj_per_inference"), "%.1f"),
                    _pair(u.get("mj_per_inference_above_idle"), p.get("mj_per_inference_above_idle"), "%.1f"),
                    _pair(u.get("avg_power_w"), p.get("avg_power_w"), "%.2f"), e["source"] or "none"))
            md.append("\nEach model runs continuously in two windows (A B B A). Energy per inference = energy "
                      "over the windows / inferences; 'above idle' subtracts the idle power measured just "
                      "before. RAPL = whole CPU package; Pi 5 PMIC = SoC and memory rails; an external USB "
                      "meter = the whole board.\n")
            for r, rt, e in sus_rows:
                if e["source"] is None:
                    warn.append("%s: %s has no energy source; add --power-csv %s=<meter log>"
                                % (exp, r["device_label"], r["device_label"]))

        md.append("## Agreement between runtimes (same model, same device, same images)\n")
        md.append("| Device | Model | Reference | Runtime | Top-1 agreement | Top-1 (ref -> runtime) |")
        md.append("|---|---|---|---|---|---|")
        for dev_label in sorted({r["device_label"] for r in rs}):
            rs_dev = sorted([r for r in rs if r["device_label"] == dev_label], key=lambda r: r["created"])
            for role in ("unpruned", "pruned"):
                ok = {}
                for r in rs_dev:
                    for x in r["runs"]:
                        if x["model"] == role and x.get("status") == "ok":
                            ok[x["runtime"]] = (x, r["eval"]["indices"])
                ref = "pytorch" if "pytorch" in ok else ("onnx" if "onnx" in ok else None)
                if ref is None:
                    continue
                xr, idx_ref = ok[ref]
                for rt, (x, idx) in ok.items():
                    if rt == ref:
                        continue
                    if idx != idx_ref:
                        warn.append("%s: %s %s and %s on %s used different images - not compared"
                                    % (exp, role, ref, rt, dev_label))
                        continue
                    ag = float(np.mean(np.asarray(x["predictions"]) == np.asarray(xr["predictions"])))
                    md.append("| %s | %s | %s | %s | %.2f%% | %.2f%% -> %.2f%% |" % (
                        dev_label, role, ref, rt, 100 * ag, 100 * xr["accuracy"]["top1"],
                        100 * x["accuracy"]["top1"]))
                    if rt != "tflite_int8" and ag < 0.995:
                        warn.append("%s: %s %s vs %s on %s agree on only %.2f%% of images (FP32 runtimes "
                                    "should match)" % (exp, role, rt, ref, dev_label, 100 * ag))

        md.append("\n## Sources\n")
        for r in rs:
            e = r["env"]
            md.append("- `%s`: device **%s**%s, %s, %s, %s logical CPUs, Python %s, threads %d, "
                      "%d warm-up + %d timed runs, %d images (%s split)%s" % (
                          r["_file"], r["device_label"], " (emulated)" if r["emulated"] else "",
                          e.get("board_model") or e.get("cpu_model") or "cpu unknown", e["machine"],
                          e["logical_cpus"], e["python"], r["settings"]["threads"], r["settings"]["warmup"],
                          r["settings"]["runs"], r["eval"]["n_used"], r["eval"]["split"],
                          " SUBSET" if r["eval"]["subset"] else ""))
        md.append("")

        if fig_rows and not a.no_figure:
            try:
                import matplotlib
                matplotlib.use("Agg")
                import matplotlib.pyplot as plt
                lbl = ["%s\n%s" % (d, rt) for d, rt, _, _ in fig_rows]
                xs = np.arange(len(fig_rows))
                fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(fig_rows)), 4.2))
                ax.bar(xs - 0.2, [u for _, _, u, _ in fig_rows], 0.4, label="unpruned", color="#9aa5b1")
                ax.bar(xs + 0.2, [p for _, _, _, p in fig_rows], 0.4, label="pruned", color="#2a6f97")
                ax.set_yscale("log")
                ax.set_ylabel("median latency, batch 1 (ms, log scale)")
                ax.set_xticks(xs)
                ax.set_xticklabels(lbl, fontsize=8)
                ax.set_title("%s: pruned vs unpruned latency" % exp)
                ax.legend(frameon=False)
                fig.tight_layout()
                fn = "%s_latency.png" % exp
                fig.savefig(os.path.join(a.out, fn), dpi=200)
                plt.close(fig)
                md.append("![latency](%s)\n" % fn)
            except ImportError:
                pass

    if warn:
        md.append("## Warnings\n")
        md += ["- " + w for w in sorted(set(warn))]
    with open(os.path.join(a.out, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    if rows_csv:
        with open(os.path.join(a.out, "report.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(rows_csv[0]))
            w.writeheader()
            w.writerows(rows_csv)
    log("wrote %s" % os.path.join(a.out, "report.md"))



_DEFAULT_GATES = {"trunk", "sdta", "heads", "mlp"}
_SHORT = {"edgenext_xx_small": "XXS", "edgenext_x_small": "XS", "edgenext_small": "S", "edgenext_base": "B"}


def _method_of(L):
    m = L["command"]
    tags = []
    if m == "gated":
        gt = set((L.get("gates") or {}).get("types") or _DEFAULT_GATES)
        if gt != _DEFAULT_GATES:
            tags.append("gates=" + ",".join(sorted(gt)))
        rt = ((L.get("export") or {}).get("projection") or {}).get("round_to")
        if rt not in (None, 8):
            tags.append("round_to=%d" % rt)
    return m + ("[" + "; ".join(tags) + "]" if tags else "")


def _ms(vals, nd=2, scale=1.0, signed=False):
    v = [x * scale for x in vals if x is not None]
    if not v:
        return "-"
    f = "%+.*f" if signed else "%.*f"
    if len(v) == 1:
        return f % (nd, v[0])
    return (f + " ± %.*f") % (nd, float(np.mean(v)), nd, float(np.std(v, ddof=1)))


def cmd_summary(a):
    """One table per question across every run: accuracy (mean ± std over
    seeds, paired difference to the same-seed baseline), complexity, training
    cost, and every device/runtime measurement, plus trade-off figures."""
    def expand(pats):
        out = []
        for pat in pats or []:
            out += glob.glob(pat, recursive=True)
        return sorted(set(out))
    log_files, res_files = expand(a.logs), expand(a.results)
    if not log_files:
        raise SystemExit("no log.json files")
    power_logs = {}
    for spec in a.power_csv or []:
        lbl, path = spec.split("=", 1)
        power_logs[lbl] = read_power_csv(path)
    os.makedirs(a.out, exist_ok=True)
    warn = []

    runs, by_sha = [], {}
    for f in log_files:
        L = read_json(f)
        if "final_eval" not in L or "checkpoint" not in L:
            warn.append("%s: unfinished run (no final_eval)" % f)
            continue
        budget = (L.get("budget") or {}).get("fraction")
        rec = {"file": f, "dataset": L["data"]["dataset"], "arch": L["arch"], "method": _method_of(L),
               "budget": 1.0 if L["command"] == "baseline" else budget, "seed": L["seed"],
               "top1": L["final_eval"]["top1"], "top5": L["final_eval"].get("top5"),
               "n_eval": L["final_eval"]["n"], "params": L["model"]["params"], "macs": L["model"]["macs"],
               "hours": L.get("wall_time_s", 0) / 3600.0, "sha": L["checkpoint"]["sha256"],
               "amp": L.get("amp")}
        if L["command"] == "gated":
            ex = L.get("export", {})
            rec["closed"] = ex.get("units_closed_by_budget_projection")
            if not ex.get("cost_model_equals_measured", True) or ex.get("gated_vs_exported_max_abs_logit_diff", 0) > 1e-4:
                warn.append("%s: export check failed (cost model / gated-vs-exported)" % f)
            if not ex.get("budget_met", True):
                warn.append("%s: exported model is over budget" % f)
        if L["command"] == "depgraph" and not (L.get("depgraph") or {}).get("budget_reachable", True):
            warn.append("%s: DepGraph could not reach budget %.2f (reached %.1f%% of MACs)" % (
                f, budget, 100 * L["depgraph"]["macs"] / L["budget"]["unpruned_macs"]))
        runs.append(rec)
        by_sha[rec["sha"]] = rec
    base = {(r["dataset"], r["arch"], r["seed"]): r for r in runs if r["method"] == "baseline"}

    groups = {}
    for r in runs:
        groups.setdefault((r["dataset"], r["arch"], r["method"], r["budget"]), []).append(r)
    order = sorted(groups, key=lambda k: (k[0], k[1], k[2] != "baseline", k[2], -(k[3] or 0)))

    md = ["# Summary of all runs\n",
          "Computed from %d training logs and %d results files; nothing is entered by hand. "
          "Mean ± standard deviation over seeds; Δ = paired difference to the baseline of the same "
          "dataset, model and seed.\n" % (len(log_files), len(res_files))]
    md.append("## Accuracy, complexity and training cost (PyTorch, full evaluation split)\n")
    md.append("| Dataset | Model | Method | MAC budget | Seeds | Params (M) | MACs (M) | Top-1 (%) | Δ top-1 (pp) | "
              "Top-5 (%) | Train time (h) |")
    md.append("|---|---|---|---|---|---|---|---|---|---|---|")
    rows_csv = []
    for k in order:
        rs = groups[k]
        deltas = [100 * (r["top1"] - base[(r["dataset"], r["arch"], r["seed"])]["top1"]) for r in rs
                  if (r["dataset"], r["arch"], r["seed"]) in base and r["method"] != "baseline"]
        md.append("| %s | %s | %s | %s | %s | %s | %s | %s | %s | %s | %s |" % (
            k[0], _SHORT.get(k[1], k[1]), k[2], "-" if k[2] == "baseline" else "%.2f" % k[3],
            ",".join(str(r["seed"]) for r in sorted(rs, key=lambda r: r["seed"])),
            _ms([r["params"] for r in rs], 3, 1e-6), _ms([r["macs"] for r in rs], 1, 1e-6),
            _ms([r["top1"] for r in rs], 2, 100), _ms(deltas, 2, signed=True) if deltas else "-",
            _ms([r["top5"] for r in rs], 2, 100), _ms([r["hours"] for r in rs], 2)))
        if len(rs) < 3:
            warn.append("%s %s %s budget %s: %d seed(s)" % (k[0], _SHORT.get(k[1], k[1]), k[2], k[3], len(rs)))
        if len({r["amp"] for r in rs + [base.get((r["dataset"], r["arch"], r["seed"])) or r for r in rs]}) > 1:
            warn.append("%s %s %s: --amp differs from its baseline" % (k[0], k[1], k[2]))
        rows_csv.append({"table": "accuracy", "dataset": k[0], "arch": k[1], "method": k[2], "budget": k[3],
                         "seeds": len(rs), "params_mean": np.mean([r["params"] for r in rs]),
                         "macs_mean": np.mean([r["macs"] for r in rs]),
                         "top1_mean": np.mean([r["top1"] for r in rs]),
                         "top1_std": np.std([r["top1"] for r in rs], ddof=1) if len(rs) > 1 else None,
                         "delta_top1_pp_mean": np.mean(deltas) if deltas else None,
                         "train_hours_mean": np.mean([r["hours"] for r in rs])})

    dev = {}
    for f in res_files:
        R = read_json(f)
        src = (R["models"].get("pruned") or {}).get("source_sha256")
        run = by_sha.get(src)
        if run is None:
            warn.append("%s: its pruned model matches no training log (pass its log.json)" % os.path.basename(f))
            continue
        for rt in ALL_RUNTIMES:
            by = {x["model"]: x for x in R["runs"] if x["runtime"] == rt and x.get("status") == "ok"}
            if "pruned" not in by or "unpruned" not in by:
                continue
            u, p = by["unpruned"], by["pruned"]
            se = sustained_energy(R, rt, power_logs) or {}
            sm = (se.get("models") or {})
            key = (run["dataset"], run["arch"], run["method"], run["budget"], R["device_label"], rt)
            dev.setdefault(key, []).append({
                "seed": run["seed"], "emulated": R["emulated"], "n": p["accuracy"]["n"],
                "top1_u": u["accuracy"]["top1"], "top1_p": p["accuracy"]["top1"],
                "lat_u": u["latency_ms"]["median"], "lat_p": p["latency_ms"]["median"],
                "speedup": u["latency_ms"]["median"] / p["latency_ms"]["median"],
                "ram_p": ((p.get("probe") or {}).get("median") or {}).get("peak_rss_bytes"),
                "ram_u": ((u.get("probe") or {}).get("median") or {}).get("peak_rss_bytes"),
                "first_p": ((p.get("probe") or {}).get("median") or {}).get("first_inference_ms"),
                "mj_p": (sm.get("pruned") or {}).get("mj_per_inference"),
                "mj_u": (sm.get("unpruned") or {}).get("mj_per_inference"),
                "tput_p": (sm.get("pruned") or {}).get("throughput_img_s")})
    if dev:
        md.append("\n## Edge-device measurements (batch 1; unpruned → pruned)\n")
        md.append("| Dataset | Model | Method | MAC budget | Device | Runtime | Seeds | Images | Top-1 (%) | "
                  "Median latency (ms) | Speedup | Peak RAM (MiB) | First inference (ms) | Energy (mJ/inference) |")
        md.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|---|")
        for key in sorted(dev, key=lambda k: (k[0], k[1], k[2], -(k[3] or 0), k[4], k[5])):
            xs = dev[key]
            fmt2 = lambda us, ps, nd, sc=1.0: "%s → %s" % (_ms(us, nd, sc), _ms(ps, nd, sc))
            md.append("| %s | %s | %s | %.2f | %s%s | %s | %d | %d | %s | %s | %s | %s | %s | %s |" % (
                key[0], _SHORT.get(key[1], key[1]), key[2], key[3], key[4],
                " (emulated)" if xs[0]["emulated"] else "", key[5], len(xs), xs[0]["n"],
                fmt2([x["top1_u"] for x in xs], [x["top1_p"] for x in xs], 2, 100),
                fmt2([x["lat_u"] for x in xs], [x["lat_p"] for x in xs], 2),
                _ms([x["speedup"] for x in xs], 2) + "x",
                fmt2([x["ram_u"] for x in xs], [x["ram_p"] for x in xs], 1, 2 ** -20),
                _ms([x["first_p"] for x in xs], 1),
                fmt2([x["mj_u"] for x in xs], [x["mj_p"] for x in xs], 1)))
            rows_csv.append({"table": "device", "dataset": key[0], "arch": key[1], "method": key[2],
                             "budget": key[3], "device": key[4], "runtime": key[5], "seeds": len(xs),
                             "latency_unpruned_ms": np.mean([x["lat_u"] for x in xs]),
                             "latency_pruned_ms": np.mean([x["lat_p"] for x in xs]),
                             "speedup_mean": np.mean([x["speedup"] for x in xs])})
        md.append("\nDevice accuracy is on the image count shown (a subset on slow devices); the table above "
                  "has the full-split accuracy. Peak RAM = whole process in a fresh run; energy needs an "
                  "on-board sensor or --power-csv.\n")

    if not a.no_figure:
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            figs = []
            all_methods = sorted({k[2] for k in groups})
            fixed = {"baseline": "#222222", "gated": "#1f77b4", "depgraph": "#ff7f0e"}
            extra = ["#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]
            colour = {}
            for m in all_methods:
                colour[m] = fixed.get(m) or extra[len([c for c in colour if c not in fixed]) % len(extra)]
            for ds_arch in sorted({(k[0], k[1]) for k in groups}):
                axes_x = [("macs", "MACs (M)", None)]
                for key in sorted({(k[4], k[5]) for k in dev if (k[0], k[1]) == ds_arch}):
                    axes_x.append(("lat", "median latency, %s / %s (ms)" % key, key))
                fig, axs = plt.subplots(1, len(axes_x), figsize=(4.6 * len(axes_x), 3.8), squeeze=False)
                for ax, (kind, xl, dk) in zip(axs[0], axes_x):
                    methods = sorted({k[2] for k in groups if (k[0], k[1]) == ds_arch})
                    for m in methods:
                        pts = []
                        for k in groups:
                            if (k[0], k[1], k[2]) != (ds_arch[0], ds_arch[1], m):
                                continue
                            rs = groups[k]
                            y = [100 * r["top1"] for r in rs]
                            if kind == "macs":
                                x = [r["macs"] / 1e6 for r in rs]
                            else:
                                xs = dev.get((k[0], k[1], k[2], k[3]) + dk)
                                if not xs:
                                    if m == "baseline":
                                        others = [v for kk, v in dev.items() if (kk[0], kk[1]) == ds_arch
                                                  and kk[4:] == dk]
                                        x = [np.mean([e["lat_u"] for v in others for e in v])] if others else []
                                    else:
                                        x = []
                                else:
                                    x = [e["lat_p"] for e in xs]
                            if x:
                                pts.append((float(np.mean(x)), float(np.mean(y)),
                                            float(np.std(y, ddof=1)) if len(y) > 1 else 0.0))
                        if not pts:
                            continue
                        pts.sort()
                        mk = "*" if m == "baseline" else "o"
                        ax.errorbar([p[0] for p in pts], [p[1] for p in pts], yerr=[p[2] for p in pts],
                                    marker=mk, ms=10 if mk == "*" else 5, capsize=3, label=m, color=colour[m],
                                    linestyle="-" if len(pts) > 1 else "none")
                    ax.set_xlabel(xl)
                    ax.set_ylabel("top-1 (%)")
                    ax.grid(alpha=0.3)
                axs[0][0].legend(frameon=False, fontsize=8)
                fig.suptitle("%s, EdgeNeXt-%s: accuracy vs cost (mean ± std over seeds)"
                             % (ds_arch[0], _SHORT.get(ds_arch[1], ds_arch[1])))
                fig.tight_layout()
                fn = "tradeoff_%s_%s.png" % ds_arch
                fig.savefig(os.path.join(a.out, fn), dpi=200)
                plt.close(fig)
                figs.append(fn)
            if figs:
                md.append("## Trade-off figures\n")
                md += ["![%s](%s)\n" % (f, f) for f in figs]
        except ImportError:
            pass

    if warn:
        md.append("## Warnings\n")
        md += ["- " + w for w in sorted(set(warn))]
    with open(os.path.join(a.out, "summary.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    if rows_csv:
        keys = []
        for r in rows_csv:
            keys += [k for k in r if k not in keys]
        with open(os.path.join(a.out, "summary.csv"), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(rows_csv)
    log("wrote %s" % os.path.join(a.out, "summary.md"))



def _depgraph_failing_group(a):
    """Rebuild the dependency graph and list the first coupled group whose index
    list does not fit its own layers (the cause of the IndexError)."""
    out = ["", "DIAGNOSIS: first dependency group whose indices exceed a member's width"]
    try:
        import timm
        import torch
        import torch_pruning as tp
        model = timm.create_model(a.arch, pretrained=False, num_classes=a.num_classes).eval()
        dg = tp.DependencyGraph().build_dependency(
            model, example_inputs=torch.randn(1, 3, a.input_size, a.input_size))
        imp_cls = getattr(tp.importance, "GroupMagnitudeImportance", None) or tp.importance.MagnitudeImportance
        imp = imp_cls(p=2)
        names = {m: n for n, m in model.named_modules()}
        for g in dg.get_all_groups(ignored_layers=[model.get_classifier()],
                                   root_module_types=[torch.nn.Conv2d, torch.nn.Linear]):
            try:
                imp(g)
                continue
            except IndexError as e:
                out.append("group fails with: %s" % e)
            for dep, idxs in g:
                layer = dep.target.module
                w = getattr(layer, "weight", None)
                if w is None and not isinstance(layer, torch.nn.Parameter):
                    continue
                shape = tuple(w.shape) if w is not None else tuple(layer.shape)
                handler = getattr(dep.handler, "__name__", type(dep.handler).__name__)
                out.append("  %-50s %-12s shape=%-18s indices=%4d (max %d)  %s" % (
                    names.get(layer, "<parameter>"), type(layer).__name__, shape, len(idxs),
                    max(idxs) if len(idxs) else -1, handler))
            break
        else:
            out.append("  no failing group found")
    except Exception as e:
        out.append("  diagnosis unavailable: %s: %s" % (type(e).__name__, e))
    return out


def cmd_depgraph(a):
    import traceback
    import timm
    import torch
    lines = ["# DepGraph (Torch-Pruning) on timm EdgeNeXt", "created: " + now_iso()]
    env = env_info()
    lines += ["%s: %s" % (k, env["packages"].get(k)) for k in ("torch", "timm", "torch-pruning")]
    lines += ["python: " + env["python"], "arch: %s  num_classes: %d  pruning_ratio: %.2f  xca_patch: %s"
              % (a.arch, a.num_classes, a.ratio, a.patch_xca), ""]
    if a.patch_xca:
        install_xca_patch()
    status = "unknown"
    try:
        import torch_pruning as tp
        torch.manual_seed(0)
        model = timm.create_model(a.arch, pretrained=False, num_classes=a.num_classes).eval()
        ex = torch.randn(1, 3, a.input_size, a.input_size)
        before = count_params(model)
        imp_cls = getattr(tp.importance, "GroupMagnitudeImportance", None) or tp.importance.MagnitudeImportance
        pruner_cls = getattr(tp.pruner, "MetaPruner", None) or tp.pruner.BasePruner
        kw = dict(importance=imp_cls(p=2), ignored_layers=[model.get_classifier()])
        try:
            pruner = pruner_cls(model, ex, pruning_ratio=a.ratio, **kw)
        except TypeError:
            pruner = pruner_cls(model, ex, ch_sparsity=a.ratio, **kw)
        lines.append("stage: pruner.step()")
        pruner.step()
        lines.append("params: %d -> %d" % (before, count_params(model)))
        lines.append("stage: forward pass after pruning")
        with torch.inference_mode():
            out = model(ex)
        lines.append("forward OK, output shape %s" % (tuple(out.shape),))
        status = "succeeded"
    except Exception:
        status = "failed"
        lines += ["", "TRACEBACK:", traceback.format_exc()]
        lines += _depgraph_failing_group(a)
    lines.insert(2, "result: " + status)
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    with open(a.out, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    log("\n".join(lines))
    log("wrote " + a.out)



def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)

    e = sub.add_parser("export", help="package pruned + unpruned for all runtimes")
    e.add_argument("--experiment", required=True, help="short tag, e.g. c10_xxs")
    e.add_argument("--dataset", required=True, choices=list(DATASETS))
    e.add_argument("--data-root", required=True)
    e.add_argument("--pruned", required=True)
    e.add_argument("--unpruned", required=True)
    e.add_argument("--out", required=True)
    e.add_argument("--arch", help="override arch inference")
    e.add_argument("--input-size", type=int, default=224)
    e.add_argument("--calib-n", type=int, default=200, help="INT8 calibration images (train split)")
    e.add_argument("--class-order", help="tiny10/tiny200: comma-separated WNIDs in label order")
    e.add_argument("--eval-per-class", type=int,
                   help="cap the evaluation set at N images per class (imagefolder/ImageNet: "
                        "use 10 for laptops, 1-2 for a 1 GB Raspberry Pi); default: whole split")
    e.add_argument("--allow-arch-mismatch", action="store_true",
                   help="reference model may be another EdgeNeXt variant (e.g. pruned S vs XS)")
    e.add_argument("--opset", type=int, default=17)
    e.add_argument("--seed", type=int, default=0)
    e.add_argument("--skip-tflite", action="store_true")
    e.add_argument("--skip-int8", action="store_true")
    e.add_argument("--download", action="store_true")
    e.set_defaults(func=cmd_export)

    b = sub.add_parser("bench", help="measure accuracy + latency on this device")
    b.add_argument("--artifacts", required=True)
    b.add_argument("--device-label", required=True, help="e.g. laptop, rpi3bplus, qemu-a72")
    b.add_argument("--runtimes", help="comma list from %s (default: all)" % ",".join(ALL_RUNTIMES))
    b.add_argument("--threads", type=int, help="default min(4, cpu count); same for every runtime")
    b.add_argument("--warmup", type=int, default=20)
    b.add_argument("--runs", type=int, default=200, help="timed runs per model")
    b.add_argument("--rounds", type=int, default=10, help="interleaving rounds")
    b.add_argument("--limit", type=int, help="class-balanced subset of about N images (slow devices)")
    b.add_argument("--per-class", type=int, help="class-balanced subset, N per class")
    b.add_argument("--emulated", choices=["auto", "yes", "no"], default="auto")
    b.add_argument("--mem-probes", type=int, default=3,
                   help="fresh-process runs per model for load time, cold-start latency and peak RAM (0 = off)")
    b.add_argument("--sustained-seconds", type=float, default=0,
                   help="continuous-inference seconds per model for sustained throughput, temperature and "
                        "energy (e.g. 60; 0 = off)")
    b.add_argument("--idle-seconds", type=float, default=10, help="idle window before the sustained runs")
    b.add_argument("--progress", type=int, default=1000)
    b.add_argument("--out")
    b.set_defaults(func=cmd_bench)

    r = sub.add_parser("report", help="combine results into one report")
    r.add_argument("--results", nargs="+", required=True)
    r.add_argument("--out", required=True)
    r.add_argument("--no-figure", action="store_true")
    r.add_argument("--power-csv", nargs="*", metavar="DEVICE_LABEL=FILE",
                   help="external power-meter log for a device (columns: unix_time, watts or volts+amps)")
    r.set_defaults(func=cmd_report)

    sm = sub.add_parser("summary", help="all runs and seeds: accuracy, cost, device metrics, trade-off figures")
    sm.add_argument("--logs", nargs="+", required=True, help="log.json files of edgenext_gated_prune.py runs")
    sm.add_argument("--results", nargs="*", help="results/*.json files of bench runs")
    sm.add_argument("--out", required=True)
    sm.add_argument("--power-csv", nargs="*", metavar="DEVICE_LABEL=FILE")
    sm.add_argument("--no-figure", action="store_true")
    sm.set_defaults(func=cmd_summary)

    pp = sub.add_parser("probe", help="internal: one model in a fresh process (called by bench)")
    pp.add_argument("--runtime", required=True, choices=ALL_RUNTIMES)
    pp.add_argument("--model-file", required=True)
    pp.add_argument("--threads", type=int, default=1)
    pp.add_argument("--arch")
    pp.add_argument("--input", required=True)
    pp.add_argument("--steady", type=int, default=10)
    pp.set_defaults(func=cmd_probe)

    d = sub.add_parser("depgraph-repro", help="log what Torch-Pruning does on timm EdgeNeXt")
    d.add_argument("--arch", default="edgenext_xx_small")
    d.add_argument("--num-classes", type=int, default=10)
    d.add_argument("--ratio", type=float, default=0.5)
    d.add_argument("--input-size", type=int, default=224)
    d.add_argument("--patch-xca", action="store_true", help="install the XCA reshape patch first")
    d.add_argument("--out", default="depgraph_failure.log")
    d.set_defaults(func=cmd_depgraph)

    args = p.parse_args(argv)
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile edgenext_gated_prune.py
"""
edgenext_gated_prune.py - coupling-typed gated structured pruning for timm
EdgeNeXt, plus the reference points a paper needs, all trained with one recipe.

Subcommands
-----------
  baseline   UNPRUNED model: phase A + phase B of the recipe, no gates.
  gated      The proposed method. Phase A trains the backbone together with
             hard-concrete (L0) gates under an augmented-Lagrangian MAC budget;
             the gates are then exported as a physically smaller timm model;
             phase B fine-tunes it.
  depgraph   DepGraph (Torch-Pruning, L2 group magnitude) restricted to the
             groups it can process on EdgeNeXt (MLP hidden units of every block
             and the stage-1 trunk). Phase A dense, one-shot prune to the same
             MAC budget, phase B fine-tune.
  check      Self-test without data (run it once on every machine): masked
             LayerNorm, gated model == exported model, cost model == measured
             MACs, rebuild from state_dict, optional ONNX / TFLite parity.

All training commands use the same data pipeline, epochs, optimiser settings
and the same single evaluation at the end (no model selection on the test
set). Each writes <out>/model.pt (portable {format, arch, num_classes,
state_dict}) and <out>/log.json. Feed model.pt files to
`edgenext_bench.py export`.

What changed with respect to the archived pruning scripts
---------------------------------------------------------
 1. All gates are sampled once per forward pass. The archived hooks drew a
    new random sample in every hook, so one stage's trunk gate had a
    different value in every block of the same forward pass.
 2. Evaluating the gated model uses the binary export mask (log_alpha > 0).
    The archived code evaluated with soft sigmoid values.
 3. LayerNorms on a pruned trunk (stem, downsample, block norms, norm_xca,
    head norm) compute their statistics over the kept channels only. In the
    archived code the pruned channels still entered the statistics, so the
    exported model computed a different function from the trained one.
 4. The trunk of the SDTA stages (stages 2-4) is gated and exported. The
    gate respects the coupling inside the split-depthwise cascade
    (sp = conv(sp + x_i) adds chunk i to chunk i-1 channel by channel): one
    "cascade" gate of the conv width, repeated for every chunk, plus a free
    gate for the pass-through chunk. The archived scripts gated these trunks
    during training but kept them full at export.
 5. The budget cost is the exact MAC count of the model the gates export
    (Conv2d + Linear + XCA matmuls, the definition of edgenext_bench.py
    count_macs), checked against a measurement. The archived proxy assumed
    7x7 kernels everywhere, ignored the stem, and counted savings that were
    never exported, so "budget met" did not mean the model got smaller.
 6. The per-branch (CAT) scalar gates are removed: they were never exported.
 7. Export copies every tensor, including the positional-encoding
    projection and all MLP biases, and is verified numerically against the
    gated model.
 8. Before export every kept width is rounded to a multiple of 8 (CPU
    kernels: arbitrary widths such as 17 or 21 erased most of the latency
    gain in ONNX Runtime) and, if the binary gates exceed the budget, the
    lowest-logit units per MAC saved are closed until the model fits. Both
    are logged (--round-to 1 disables rounding).
 9. Taylor sensitivities come from the previous steps (EMA) so each step
    needs one backward pass (no retain_graph).
10. No test-set evaluation during training; one evaluation before phase B
    (reported, not used for decisions) and one at the end.

Example (CIFAR-10, EdgeNeXt-XXS, 50 % MAC budget, seed 0)
--------------------------------------------------------
  python edgenext_gated_prune.py check
  python edgenext_gated_prune.py baseline --arch edgenext_xx_small --dataset cifar10 \\
      --data-root ./data --seed 0 --out runs/c10_xxs/unpruned_s0
  python edgenext_gated_prune.py gated --arch edgenext_xx_small --dataset cifar10 \\
      --data-root ./data --budget 0.5 --seed 0 --out runs/c10_xxs/gated_b50_s0
  python edgenext_gated_prune.py depgraph --arch edgenext_xx_small --dataset cifar10 \\
      --data-root ./data --budget 0.5 --seed 0 --out runs/c10_xxs/depgraph_b50_s0

Long runs can be continued after a disconnect with the same command plus
--resume.
"""

import argparse
import copy
import math
import os
import random
import sys
import time
import types

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)
import edgenext_bench as eb

GAMMA, ZETA = -0.1, 1.1
GATE_TYPES = ("trunk", "sdta", "heads", "mlp")

RECIPES = {
    "xxs": {"epochs_a": 15, "epochs_b": 10, "batch_size": {"cifar10": 32, "cifar100": 32}, "batch_default": 64,
            "opt_a": "adam", "lr_a": 1e-4, "wd_a": 0.0, "cosine_a": False, "label_smoothing": 0.0,
            "clip_a": 0.0, "lr_b": 1e-2, "nesterov_b": False, "clip_b": 5.0, "warmup_b": 1.0, "strong_aug": False,
            "gate_lr": 1e-3, "warmup_epochs": 0, "eta": 0.5, "lambda_max": 50.0, "rho_start": 0.1,
            "rho_end": 20.0, "beta_start": 2.0 / 3.0, "beta_end": 0.1, "alpha": 1e-3, "budget": 0.5},
    "s": {"epochs_a": 30, "epochs_b": 20, "batch_size": {}, "batch_default": 64,
          "opt_a": "adamw", "lr_a": 5e-4, "wd_a": 5e-4, "cosine_a": True, "label_smoothing": 0.1,
          "clip_a": 1.0, "lr_b": 3e-2, "nesterov_b": True, "clip_b": 5.0, "warmup_b": 1.0, "strong_aug": True,
          "gate_lr": 1e-2, "warmup_epochs": 5, "eta": 0.3, "lambda_max": 30.0, "rho_start": 0.05,
          "rho_end": 15.0, "beta_start": 2.0 / 3.0, "beta_end": 0.1, "alpha": 1e-4, "budget": 0.7},
}


def log(msg):
    eb.log(msg)



def hc_sample(log_alpha, beta):
    import torch
    u = torch.rand_like(log_alpha).clamp(1e-6, 1 - 1e-6)
    s = torch.sigmoid((torch.log(u) - torch.log(1 - u) + log_alpha) / beta)
    return (s * (ZETA - GAMMA) + GAMMA).clamp(0.0, 1.0)


def hard_mask(log_alpha, min_keep=1):
    """Binary export mask: stretched sigmoid > 0.5, i.e. log_alpha > 0. At least
    min_keep units stay open (the highest logits)."""
    import torch
    la = log_alpha.detach()
    m = (la > 0).to(la.dtype)
    if int(m.sum()) < min_keep:
        m[torch.topk(la, min_keep).indices] = 1.0
    return m


def masked_layer_norm(x, mask, weight, bias, eps, dim):
    """LayerNorm over `dim` using only channels where mask > 0 (weighted by the
    mask when it is soft); output multiplied by the mask. With a binary mask
    this equals LayerNorm on the kept channels and zero elsewhere."""
    import torch
    dt = x.dtype
    x = x.float()
    shape = [1] * x.dim()
    shape[dim] = -1
    m = mask.float().view(shape)
    denom = mask.float().sum().clamp_min(1e-6)
    mean = (x * m).sum(dim, keepdim=True) / denom
    xc = x - mean
    var = (xc * xc * m).sum(dim, keepdim=True) / denom
    y = xc * torch.rsqrt(var + eps) * weight.float().view(shape) + bias.float().view(shape)
    return (y * m).to(dt)


class GatedEdgeNeXt:
    """Wraps a timm EdgeNeXt with coupling-typed gates (hooks + LayerNorm
    patches). Built on CPU, then moved with .to(device).

    Gate groups (one logit per unit, init_logit = open):
      trunk_s{s}            stage trunk of a stage without SDTA block  [C]
      casc_s{s}, tail_s{s}  stage trunk of an SDTA stage: cascade columns [w]
                            (repeated for each of the n conv chunks) and the
                            pass-through chunk [C - n*w]
      head_s{s}_b{b}        XCA heads of an SDTA block                 [H]
      mlp_s{s}_b{b}         MLP hidden units of every block            [E]
    """

    def __init__(self, model, arch, num_classes, input_size=224, gate_types=GATE_TYPES,
                 init_logit=2.0, sens_decay=0.9):
        import torch
        import torch.nn as nn
        self.model = model
        self.arch = arch
        self.num_classes = int(num_classes)
        self.gate_types = tuple(gate_types)
        self.beta = 2.0 / 3.0
        self.training = True
        self.sens_decay = sens_decay
        self.grad_scale = 1.0
        self.sens = {}
        self._z = {}
        self._trunk = [None] * len(model.stages)
        self._handles = []
        self._ln_patched = []
        self.stages = self._describe(input_size)
        self.gates = nn.ParameterDict()
        for s, st in enumerate(self.stages):
            if st["mode"] == "free" and "trunk" in self.gate_types:
                self.gates["trunk_s%d" % s] = nn.Parameter(torch.full((st["dim"],), init_logit))
            if st["mode"] == "sdta" and "sdta" in self.gate_types:
                self.gates["casc_s%d" % s] = nn.Parameter(torch.full((st["w"],), init_logit))
                self.gates["tail_s%d" % s] = nn.Parameter(torch.full((st["dim"] - st["n"] * st["w"],), init_logit))
            for b, bi in enumerate(st["blocks"]):
                if bi["type"] == "sdta" and "heads" in self.gate_types:
                    self.gates["head_s%d_b%d" % (s, b)] = nn.Parameter(torch.full((bi["H"],), init_logit))
                if "mlp" in self.gate_types:
                    self.gates["mlp_s%d_b%d" % (s, b)] = nn.Parameter(torch.full((bi["E"],), init_logit))
        self._install()


    def _describe(self, input_size):
        import torch
        import torch.nn as nn
        m = self.model
        shapes = {}
        hs = [st.register_forward_hook(lambda mod, i, o, s=s: shapes.__setitem__(s, tuple(o.shape)))
              for s, st in enumerate(m.stages)]
        was = m.training
        m.eval()
        with torch.no_grad():
            m(torch.zeros(1, 3, input_size, input_size, device=next(m.parameters()).device))
        m.train(was)
        for h in hs:
            h.remove()
        conv = m.stem[0]
        self.stem_k = int(conv.in_channels // conv.groups * conv.kernel_size[0] * conv.kernel_size[1])
        stages = []
        for s, st in enumerate(m.stages):
            _, C, H, W = shapes[s]
            info = {"dim": int(C), "N": int(H * W), "ds_k": None, "blocks": [], "mode": "free", "n": 0, "w": 0}
            if not isinstance(st.downsample, nn.Identity):
                dc = st.downsample[1]
                info["ds_k"] = int(dc.kernel_size[0] * dc.kernel_size[1])
            for blk in st.blocks:
                bi = {"E": int(blk.mlp.fc1.out_features)}
                if hasattr(blk, "xca"):
                    n, w = len(blk.convs), int(blk.convs[0].in_channels)
                    heads = int(blk.xca.num_heads)
                    bi.update(type="sdta", n=n, w=w, H=heads, hd=int(blk.xca.qkv.out_features) // 3 // heads,
                              ck2=int(blk.convs[0].kernel_size[0] * blk.convs[0].kernel_size[1]),
                              pos_in=int(blk.pos_embd.token_projection.in_channels) if blk.pos_embd is not None else 0)
                    if info["mode"] == "sdta" and (info["n"], info["w"]) != (n, w):
                        raise ValueError("SDTA blocks of stage %d disagree on their split" % s)
                    info.update(mode="sdta", n=n, w=w)
                else:
                    bi.update(type="conv", k2=int(blk.conv_dw.kernel_size[0] * blk.conv_dw.kernel_size[1]))
                info["blocks"].append(bi)
            stages.append(info)
        return stages


    def to(self, dev):
        self.model.to(dev)
        self.gates.to(dev)
        return self

    def train(self, mode=True):
        self.training = mode
        self.model.train(mode)
        return self

    def eval(self):
        return self.train(False)

    def parameters_backbone(self):
        return self.model.parameters()

    def parameters_gates(self):
        return self.gates.parameters()

    def state_dict(self):
        return {"model": self.model.state_dict(), "gates": {k: v.detach().cpu() for k, v in self.gates.items()},
                "sens": {k: v.detach().cpu() for k, v in self.sens.items()}}

    def load_state_dict(self, sd):
        import torch
        self.model.load_state_dict(sd["model"])
        with torch.no_grad():
            for k, v in sd["gates"].items():
                self.gates[k].copy_(v)
        dev = next(self.model.parameters()).device
        self.sens = {k: v.to(dev) for k, v in sd.get("sens", {}).items()}

    def __call__(self, x):
        self.sample()
        return self.model(x)


    def sample(self):
        z = {}
        for k, la in self.gates.items():
            z[k] = hc_sample(la, self.beta) if self.training else hard_mask(la)
        self._z = z
        self._trunk = [self._build_trunk(s, z) for s in range(len(self.stages))]

    def _build_trunk(self, s, z):
        import torch
        st = self.stages[s]
        if st["mode"] == "free":
            return z.get("trunk_s%d" % s)
        c = z.get("casc_s%d" % s)
        if c is None:
            return None
        return torch.cat([c] * st["n"] + [z["tail_s%d" % s]])

    def hard_masks(self):
        return {k: hard_mask(la).cpu() for k, la in self.gates.items()}


    def _patch_ln(self, mod, getter, dim):
        orig = type(mod).forward

        def fwd(this, x, getter=getter, dim=dim, orig=orig):
            m = getter()
            if m is None:
                return orig(this, x)
            return masked_layer_norm(x, m, this.weight, this.bias, this.eps, dim)
        mod.forward = types.MethodType(fwd, mod)
        self._ln_patched.append(mod)

    def _install(self):
        import torch.nn as nn
        m = self.model
        self._patch_ln(m.stem[1], lambda: self._trunk[0], 1)
        for s, st in enumerate(m.stages):
            if not isinstance(st.downsample, nn.Identity):
                self._patch_ln(st.downsample[0], lambda s=s: self._trunk[s - 1], 1)
                self._handles.append(st.downsample.register_forward_hook(self._trunk_hook(s)))
            for b, blk in enumerate(st.blocks):
                self._patch_ln(blk.norm, lambda s=s: self._trunk[s], -1)
                if hasattr(blk, "xca"):
                    self._patch_ln(blk.norm_xca, lambda s=s: self._trunk[s], -1)
                    self._handles.append(blk.xca.proj.register_forward_pre_hook(self._head_hook(s, b)))
                self._handles.append(blk.mlp.fc1.register_forward_hook(self._mlp_hook(s, b)))
                self._handles.append(blk.register_forward_hook(self._trunk_hook(s)))
        self._patch_ln(m.head.norm, lambda: self._trunk[-1], 1)

    def remove_hooks(self):
        for h in self._handles:
            h.remove()
        self._handles = []
        for mod in self._ln_patched:
            if "forward" in mod.__dict__:
                del mod.forward
        self._ln_patched = []

    def _trunk_hook(self, s):
        def hook(_m, _i, out):
            t = self._trunk[s]
            return out if t is None else out * t.view(1, -1, 1, 1).to(out.dtype)
        return hook

    def _head_hook(self, s, b):
        key = "head_s%d_b%d" % (s, b)
        H = self.stages[s]["blocks"][b]["H"]

        def hook(_m, inputs):
            z = self._z.get(key)
            if z is None:
                return None
            x = inputs[0]
            B, N, C = x.shape
            return ((x.view(B, N, H, C // H) * z.view(1, 1, H, 1).to(x.dtype)).reshape(B, N, C),)
        return hook

    def _mlp_hook(self, s, b):
        key = "mlp_s%d_b%d" % (s, b)

        def hook(_m, _i, out):
            z = self._z.get(key)
            if z is None:
                return out
            y = out * z.to(out.dtype)
            if self.training and y.requires_grad:
                act = y.detach()

                def grad_hook(g, act=act):
                    p = (act.float() * g.float() / self.grad_scale) ** 2
                    score = p.mean(dim=tuple(range(p.dim() - 1))).detach()
                    old = self.sens.get(key)
                    self.sens[key] = score if old is None else self.sens_decay * old + (1 - self.sens_decay) * score
                y.register_hook(grad_hook)
            return y
        return hook


    def cost_from_counts(self, c):
        """Exact MACs of the exported model given unit counts per gate group
        (python ints) or their soft sums (tensors, differentiable)."""
        T = []
        for s, st in enumerate(self.stages):
            if st["mode"] == "free" and ("trunk_s%d" % s) in c:
                T.append(c["trunk_s%d" % s])
            elif st["mode"] == "sdta" and ("casc_s%d" % s) in c:
                T.append(st["n"] * c["casc_s%d" % s] + c["tail_s%d" % s])
            else:
                T.append(st["dim"])
        total = self.stem_k * T[0] * self.stages[0]["N"]
        for s, st in enumerate(self.stages):
            N, Ts = st["N"], T[s]
            if s > 0:
                total = total + st["ds_k"] * T[s - 1] * Ts * N
            for b, bi in enumerate(st["blocks"]):
                E = c.get("mlp_s%d_b%d" % (s, b), bi["E"])
                total = total + 2 * N * Ts * E
                if bi["type"] == "conv":
                    total = total + bi["k2"] * Ts * N
                else:
                    Wc = c.get("casc_s%d" % s, bi["w"])
                    total = total + bi["n"] * bi["ck2"] * Wc * N
                    if bi["pos_in"]:
                        total = total + bi["pos_in"] * Ts * N
                    Hk = c.get("head_s%d_b%d" % (s, b), bi["H"])
                    total = total + 4 * N * Ts * Hk * bi["hd"] + 2 * Hk * bi["hd"] * bi["hd"] * N
        return total + T[-1] * self.num_classes

    def soft_cost(self):
        return self.cost_from_counts({k: v.sum() for k, v in self._z.items()})

    def hard_counts(self):
        return {k: int(v.sum()) for k, v in self.hard_masks().items()}

    def hard_cost(self):
        return int(self.cost_from_counts(self.hard_counts()))

    def full_cost(self):
        return int(self.cost_from_counts({k: int(v.numel()) for k, v in self.gates.items()}))

    def sens_reg(self):
        reg = 0.0
        for k, imp in self.sens.items():
            z = self._z.get(k)
            if z is not None:
                reg = reg + ((1 - z) * imp.to(z.device)).sum()
        return reg

    def fit_budget(self, budget_macs, round_to=8):
        """Make the binary gates exportable as an efficient model within the
        budget: (1) every group's kept count is rounded to the nearest multiple
        of round_to (or the group's full size; head counts are not rounded);
        (2) while over budget, the group whose next step down closes the
        lowest-logit units per MAC saved is reduced by one step; (3) slack left
        by (2) is given back to the highest-logit units. Kept units are always
        the highest-logit ones of their group. Returns units closed/opened."""
        import torch
        la = {k: v.detach().cpu().clone() for k, v in self.gates.items()}
        sizes = {k: int(v.numel()) for k, v in la.items()}
        order = {k: torch.argsort(v, descending=True) for k, v in la.items()}
        before = {k: int(v.sum()) for k, v in self.hard_masks().items()}
        r = max(1, int(round_to))

        def allowed(k):
            n = sizes[k]
            if k.startswith("head") or r == 1 or n < r:
                return list(range(1, n + 1))
            vals = list(range(r, n + 1, r))
            return vals if vals[-1] == n else vals + [n]

        al = {k: allowed(k) for k in la}
        counts = {k: min(al[k], key=lambda v, c=before[k]: (abs(v - c), -v)) for k in la}
        cost = self.cost_from_counts(counts)
        while cost > budget_macs:
            best = None
            for k in counts:
                lower = [v for v in al[k] if v < counts[k]]
                if not lower:
                    continue
                nv = lower[-1]
                trial = dict(counts)
                trial[k] = nv
                c2 = self.cost_from_counts(trial)
                saved = cost - c2
                if saved <= 0:
                    continue
                removed = order[k][nv:counts[k]]
                score = float(la[k][removed].sum()) / saved
                if best is None or score < best[0]:
                    best = (score, k, nv, c2)
            if best is None:
                break
            _, k, nv, cost = best
            counts[k] = nv
        target = min(budget_macs, self.cost_from_counts(before))
        while True:
            best = None
            for k in counts:
                higher = [v for v in al[k] if v > counts[k]]
                if not higher:
                    continue
                nv = higher[0]
                trial = dict(counts)
                trial[k] = nv
                c2 = self.cost_from_counts(trial)
                if c2 > target or c2 <= cost:
                    continue
                added = order[k][counts[k]:nv]
                score = float(la[k][added].sum()) / (c2 - cost)
                if best is None or score > best[0]:
                    best = (score, k, nv, c2)
            if best is None:
                break
            _, k, nv, cost = best
            counts[k] = nv
        closed = opened = 0
        with torch.no_grad():
            for k, g in self.gates.items():
                keep = order[k][:counts[k]]
                newla = torch.full_like(la[k], -10.0)
                newla[keep] = torch.clamp(la[k][keep], min=1.0)
                closed += max(0, before[k] - counts[k])
                opened += max(0, counts[k] - before[k])
                g.copy_(newla.to(g.device))
        return {"units_closed": int(closed), "units_opened": int(opened), "round_to": r,
                "cost_macs": int(self.hard_cost()), "fits": self.hard_cost() <= budget_macs}



def export_pruned(g):
    """Physically pruned timm EdgeNeXt (CPU, eval) equal to g in eval mode."""
    import timm
    import torch
    import torch.nn as nn
    eb.install_edgenext_patches()
    old = g.model
    masks = g.hard_masks()
    idx = {k: torch.nonzero(m).flatten() for k, m in masks.items()}
    trunk = []
    for s, st in enumerate(g.stages):
        if st["mode"] == "free" and ("trunk_s%d" % s) in idx:
            trunk.append(idx["trunk_s%d" % s])
        elif st["mode"] == "sdta" and ("casc_s%d" % s) in idx:
            c, t = idx["casc_s%d" % s], idx["tail_s%d" % s]
            trunk.append(torch.cat([c + i * st["w"] for i in range(st["n"])] + [t + st["n"] * st["w"]]))
        else:
            trunk.append(torch.arange(st["dim"]))
    dims = tuple(int(t.numel()) for t in trunk)
    new = timm.create_model(g.arch, pretrained=False, num_classes=g.num_classes, dims=dims)

    def P(t):
        return t.detach().cpu()

    def copy_vec(dst, src, ix):
        dst.data.copy_(P(src)[ix])

    def linear(src, rows, cols):
        w = P(src.weight)
        w = w[rows] if rows is not None else w
        w = w[:, cols] if cols is not None else w
        lin = nn.Linear(int(w.shape[1]), int(w.shape[0]), bias=src.bias is not None)
        lin.weight.data.copy_(w)
        if src.bias is not None:
            lin.bias.data.copy_(P(src.bias)[rows] if rows is not None else P(src.bias))
        return lin

    with torch.no_grad():
        i0 = trunk[0]
        new.stem[0].weight.copy_(P(old.stem[0].weight)[i0])
        if old.stem[0].bias is not None:
            copy_vec(new.stem[0].bias, old.stem[0].bias, i0)
        copy_vec(new.stem[1].weight, old.stem[1].weight, i0)
        copy_vec(new.stem[1].bias, old.stem[1].bias, i0)
        for s, (sn, so) in enumerate(zip(new.stages, old.stages)):
            it = trunk[s]
            if s > 0 and not isinstance(so.downsample, nn.Identity):
                ip = trunk[s - 1]
                copy_vec(sn.downsample[0].weight, so.downsample[0].weight, ip)
                copy_vec(sn.downsample[0].bias, so.downsample[0].bias, ip)
                sn.downsample[1].weight.copy_(P(so.downsample[1].weight)[it][:, ip])
                if so.downsample[1].bias is not None:
                    copy_vec(sn.downsample[1].bias, so.downsample[1].bias, it)
            for b, (bn, bo) in enumerate(zip(sn.blocks, so.blocks)):
                e = idx.get("mlp_s%d_b%d" % (s, b), torch.arange(bo.mlp.fc1.out_features))
                bn.mlp.fc1 = linear(bo.mlp.fc1, e, it)
                bn.mlp.fc2 = linear(bo.mlp.fc2, it, e)
                copy_vec(bn.norm.weight, bo.norm.weight, it)
                copy_vec(bn.norm.bias, bo.norm.bias, it)
                if bo.gamma is not None:
                    copy_vec(bn.gamma, bo.gamma, it)
                if not hasattr(bo, "xca"):
                    bn.conv_dw.weight.copy_(P(bo.conv_dw.weight)[it])
                    if bo.conv_dw.bias is not None:
                        copy_vec(bn.conv_dw.bias, bo.conv_dw.bias, it)
                    continue
                st = g.stages[s]
                ci = idx.get("casc_s%d" % s, torch.arange(st["w"]))
                convs = []
                for co in bo.convs:
                    k = int(co.kernel_size[0])
                    cv = nn.Conv2d(int(ci.numel()), int(ci.numel()), k, padding=k // 2,
                                   groups=int(ci.numel()), bias=co.bias is not None)
                    cv.weight.copy_(P(co.weight)[ci])
                    if co.bias is not None:
                        cv.bias.copy_(P(co.bias)[ci])
                    convs.append(cv)
                bn.convs = nn.ModuleList(convs)
                bn.width = int(ci.numel())
                if bo.pos_embd is not None:
                    tp_new, tp_old = bn.pos_embd.token_projection, bo.pos_embd.token_projection
                    tp_new.weight.copy_(P(tp_old.weight)[it])
                    copy_vec(tp_new.bias, tp_old.bias, it)
                copy_vec(bn.norm_xca.weight, bo.norm_xca.weight, it)
                copy_vec(bn.norm_xca.bias, bo.norm_xca.bias, it)
                if bo.gamma_xca is not None:
                    copy_vec(bn.gamma_xca, bo.gamma_xca, it)
                H = int(bo.xca.num_heads)
                inner = int(bo.xca.qkv.out_features) // 3
                hd = inner // H
                hk = idx.get("head_s%d_b%d" % (s, b), torch.arange(H))
                qsel = torch.cat([torch.arange(h * hd, (h + 1) * hd) for h in hk.tolist()])
                bn.xca.qkv = linear(bo.xca.qkv, torch.cat([qsel, qsel + inner, qsel + 2 * inner]), it)
                bn.xca.proj = linear(bo.xca.proj, it, qsel)
                bn.xca.num_heads = int(hk.numel())
                bn.xca.temperature = nn.Parameter(P(bo.xca.temperature)[hk].clone())
        il = trunk[-1]
        copy_vec(new.head.norm.weight, old.head.norm.weight, il)
        copy_vec(new.head.norm.bias, old.head.norm.bias, il)
        new.head.fc.weight.copy_(P(old.head.fc.weight)[:, il])
        new.head.fc.bias.copy_(P(old.head.fc.bias))
    return new.eval()


def compare_outputs(a, b, x):
    import torch
    with torch.no_grad():
        return float((a(x).float().cpu() - b(x).float().cpu()).abs().max())



def train_transform(dataset, size, strong):
    from torchvision import transforms as T
    norm = T.Normalize(eb.IMAGENET_MEAN, eb.IMAGENET_STD)
    if dataset in ("cifar10", "cifar100"):
        base = [T.Resize((size, size)), T.RandomCrop(size, padding=size // 8 if strong else 8),
                T.RandomHorizontalFlip()]
    elif dataset in ("tiny10", "tiny200"):
        base = [T.RandomResizedCrop(size, scale=(0.70, 1.0)), T.RandomHorizontalFlip()]
    else:
        base = [T.RandomResizedCrop(size), T.RandomHorizontalFlip()]
    if strong:
        return T.Compose(base + [T.RandAugment(num_ops=2, magnitude=9), T.ToTensor(),
                                 T.RandomErasing(p=0.25), norm])
    return T.Compose(base + [T.ToTensor(), norm])


class ArrayTrain:
    """uint8 HWC images in memory -> augmented tensors (module level so it can
    be pickled by DataLoader workers on every OS)."""
    def __init__(self, imgs, labels, tf):
        self.imgs, self.labels, self.tf = imgs, labels, tf

    def __len__(self):
        return int(len(self.labels))

    def __getitem__(self, i):
        from PIL import Image
        return self.tf(Image.fromarray(self.imgs[i])), int(self.labels[i])


class ArrayEval:
    def __init__(self, imgs, labels, cfg):
        self.imgs, self.labels, self.cfg = imgs, labels, cfg

    def __len__(self):
        return int(len(self.labels))

    def __getitem__(self, i):
        import torch
        return torch.from_numpy(eb.preprocess(self.imgs[i], self.cfg)), int(self.labels[i])


class PathTrain:
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = paths, labels, tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        from PIL import Image
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert("RGB")), int(self.labels[i])


class PathEval:
    def __init__(self, paths, labels, cfg):
        self.paths, self.labels, self.cfg = paths, labels, cfg

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        import torch
        return torch.from_numpy(eb.preprocess(eb.load_presized(self.paths[i], self.cfg), self.cfg)), \
            int(self.labels[i])


def build_data(a, rc):
    """(train_set, eval_set, info). Evaluation preprocessing is the function
    edgenext_bench.py uses on every device."""
    ds = eb.DATASETS[a.dataset]
    cfg = dict(ds["preprocess"])
    cfg["size"] = a.input_size
    tf = train_transform(a.dataset, a.input_size, rc["strong_aug"])
    class_order = a.class_order.split(",") if a.class_order else None
    if a.dataset == "imagefolder":
        tr_paths, tr_labels, names = eb.list_imagefolder(a.data_root, "train", a.train_per_class)
        ev_paths, ev_labels, _ = eb.list_imagefolder(a.data_root, "val", a.eval_per_class)
        train, evals = PathTrain(tr_paths, tr_labels, tf), PathEval(ev_paths, ev_labels, cfg)
    else:
        tr_imgs, tr_labels, names = eb.load_raw_split(a.dataset, a.data_root, "train", class_order,
                                                      a.download, per_class=a.train_per_class)
        if a.dataset in ("tiny10", "tiny200"):
            class_order = names
        ev_imgs, ev_labels, _ = eb.load_raw_split(a.dataset, a.data_root, ds["eval_split"], class_order,
                                                  a.download, per_class=a.eval_per_class)
        train, evals = ArrayTrain(tr_imgs, tr_labels, tf), ArrayEval(ev_imgs, ev_labels, cfg)
    info = {"dataset": a.dataset, "num_classes": eb.dataset_num_classes(a.dataset, a.data_root),
            "class_names": list(names) if names is not None else None, "eval_split": ds["eval_split"],
            "n_train": len(tr_labels), "n_eval": len(ev_labels), "train_per_class_cap": a.train_per_class,
            "eval_per_class_cap": a.eval_per_class, "eval_preprocess": cfg,
            "train_transform": repr(tf)}
    return train, evals, info


def _worker_init(wid):
    import torch
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


def make_loader(dset, bs, workers, seed, shuffle, dev, drop_last):
    import torch
    g = torch.Generator()
    g.manual_seed(seed)
    return torch.utils.data.DataLoader(dset, batch_size=bs, shuffle=shuffle, drop_last=drop_last,
                                       num_workers=workers, pin_memory=dev.type == "cuda",
                                       worker_init_fn=_worker_init, generator=g)


def evaluate(model, loader, dev, amp):
    import torch
    model.eval()
    correct, correct5, n = 0, 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=amp):
                logits = model(x).float().cpu()
            top = logits.topk(min(5, logits.shape[1]), dim=1).indices
            correct += int((top[:, 0] == y).sum())
            correct5 += int((top == y[:, None]).any(1).sum())
            n += int(y.numel())
    return {"correct": correct, "n": n, "top1": correct / max(n, 1),
            "top5_correct": correct5, "top5": correct5 / max(n, 1)}



def seed_all(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed % 2 ** 32)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def recipe_for(a):
    rc = dict(RECIPES[a.recipe or ("xxs" if a.arch == "edgenext_xx_small" else "s")])
    for kv in a.override or []:
        k, v = kv.split("=", 1)
        if k not in rc:
            raise SystemExit("unknown recipe key %r (keys: %s)" % (k, ", ".join(sorted(rc))))
        old = rc[k]
        rc[k] = (v.lower() in ("1", "true", "yes")) if isinstance(old, bool) else type(old)(v) \
            if not isinstance(old, dict) else old
    if a.epochs_a is not None:
        rc["epochs_a"] = a.epochs_a
    if a.epochs_b is not None:
        rc["epochs_b"] = a.epochs_b
    if getattr(a, "budget", None) is not None:
        rc["budget"] = a.budget
    if getattr(a, "warmup_epochs", None) is not None:
        rc["warmup_epochs"] = a.warmup_epochs
    rc["batch"] = a.batch_size or rc["batch_size"].get(a.dataset, rc["batch_default"])
    return rc


class Run:
    def __init__(self, out):
        self.out = os.path.abspath(out)
        os.makedirs(self.out, exist_ok=True)
        self.resume_path = os.path.join(self.out, "resume.pt")

    def save(self, state):
        import torch
        tmp = self.resume_path + ".tmp"
        torch.save(state, tmp)
        os.replace(tmp, self.resume_path)

    def load(self):
        import torch
        if os.path.exists(self.resume_path):
            return torch.load(self.resume_path, map_location="cpu", weights_only=False)
        return None

    def finish(self, keep):
        if not keep and os.path.exists(self.resume_path):
            os.remove(self.resume_path)


def common_log(a, rc, info, dev, command):
    return {"schema": "edgenext-prune/1", "command": command, "argv": sys.argv, "created": eb.now_iso(),
            "env": eb.env_info(), "script_sha256": eb.sha256_file(os.path.abspath(__file__)),
            "bench_sha256": eb.sha256_file(os.path.abspath(eb.__file__)),
            "arch": a.arch, "seed": a.seed, "pretrained": not a.no_pretrained, "device": str(dev),
            "amp": bool(a.amp), "recipe": a.recipe or ("xxs" if a.arch == "edgenext_xx_small" else "s"),
            "recipe_values": {k: v for k, v in rc.items() if k != "batch_size"}, "data": info, "history": []}


def save_model(model, arch, path):
    eb.save_portable_checkpoint(model.cpu().eval(), arch, path)
    return {"file": os.path.basename(path), "sha256": eb.sha256_file(path), "bytes": os.path.getsize(path)}


def complexity(model, input_size):
    m = copy.deepcopy(model).cpu().eval()
    return {"params": eb.count_params(m), "macs": eb.count_macs(m, input_size)["macs"],
            "arch_summary": eb.arch_summary(m)}


def setup(a, command):
    import torch
    eb.install_edgenext_patches()
    rc = recipe_for(a)
    dev = torch.device("cuda" if torch.cuda.is_available() and not a.cpu else "cpu")
    if a.amp and dev.type != "cuda":
        log("--amp ignored on CPU")
        a.amp = False
    seed_all(a.seed)
    train_set, eval_set, info = build_data(a, rc)
    log("%s: %s on %s, %d train / %d %s images, %d classes, batch %d, device %s"
        % (command, a.arch, a.dataset, info["n_train"], info["n_eval"], info["eval_split"],
           info["num_classes"], rc["batch"], dev))
    eval_loader = make_loader(eval_set, 128, a.workers, a.seed, False, dev, False)
    return rc, dev, train_set, eval_loader, info


def new_model(a, num_classes):
    import timm
    return timm.create_model(a.arch, pretrained=not a.no_pretrained, num_classes=num_classes)


def epoch_loader(train_set, rc, a, dev, phase, epoch):
    seed_all(a.seed * 1000 + (0 if phase == "A" else 500) + epoch)
    return make_loader(train_set, rc["batch"], a.workers, a.seed * 1000 + epoch, True, dev, True)


def train_epoch_dense(model, loader, opt, dev, ls, clip, scaler, amp, lr_fn=None, step0=0):
    import torch
    import torch.nn.functional as F
    model.train()
    tot, n = 0.0, 0
    for i, (x, y) in enumerate(loader):
        if lr_fn is not None:
            for g in opt.param_groups:
                g["lr"] = lr_fn(step0 + i)
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=dev.type, enabled=amp):
            loss = F.cross_entropy(model(x), y, label_smoothing=ls)
        scaler.scale(loss).backward()
        if clip > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(opt)
        scaler.update()
        tot += float(loss.detach()) * x.size(0)
        n += x.size(0)
    return tot / max(n, 1)


def make_opt_a(params_groups, rc, epochs):
    import torch
    cls = torch.optim.Adam if rc["opt_a"] == "adam" else torch.optim.AdamW
    opt = cls(params_groups)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=rc["lr_a"] * 0.01) \
        if rc["cosine_a"] else None
    return opt, sched


def dense_phase_a(model, a, rc, dev, train_set, run, state, lg, arch):
    """Phase A without gates (baseline, depgraph). Resumable."""
    import torch
    opt, sched = make_opt_a([{"params": model.parameters(), "lr": rc["lr_a"], "weight_decay": rc["wd_a"]}],
                            rc, rc["epochs_a"])
    scaler = torch.amp.GradScaler("cuda", enabled=a.amp)
    start = 0
    if state and state.get("phase") == "A":
        opt.load_state_dict(state["opt"])
        if sched and state.get("sched"):
            sched.load_state_dict(state["sched"])
        scaler.load_state_dict(state["scaler"])
        start = state["epoch"]
    for e in range(start, rc["epochs_a"]):
        t0 = time.time()
        loss = train_epoch_dense(model, epoch_loader(train_set, rc, a, dev, "A", e), opt, dev,
                                 rc["label_smoothing"], rc["clip_a"], scaler, a.amp)
        if sched:
            sched.step()
        lg["history"].append({"phase": "A", "epoch": e + 1, "train_loss": loss, "seconds": time.time() - t0})
        log("[A %d/%d] loss %.4f (%.0fs)" % (e + 1, rc["epochs_a"], loss, time.time() - t0))
        run.save({"phase": "A", "epoch": e + 1, "model_sd": model.state_dict(), "arch": arch,
                  "opt": opt.state_dict(), "sched": sched.state_dict() if sched else None,
                  "scaler": scaler.state_dict(), "log": lg})


def start_train_loss(model, train_set, rc, a, dev, batches=20):
    """Mean training loss (eval mode, no update) on the first `batches` batches of a
    fixed-seed loader. Training batches are reseeded per epoch, so this extra pass
    does not change any later batch."""
    import torch
    import torch.nn.functional as F
    model.eval()
    loader = make_loader(train_set, rc["batch"], a.workers, a.seed * 1000 + 999, True, dev, True)
    tot, n = 0.0, 0
    with torch.inference_mode():
        for i, (x, y) in enumerate(loader):
            if i >= batches:
                break
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=a.amp):
                out = model(x)
            tot += float(F.cross_entropy(out.float(), y)) * x.size(0)
            n += x.size(0)
    del loader
    return tot / max(n, 1)


def phase_b(model, a, rc, dev, train_set, run, state, lg, arch, extra=None):
    """Post-export fine-tune, identical for every method: SGD, linear warm-up
    for warmup_b epochs, then cosine to zero, per step. Resumable. Stops early
    if the first epoch collapses to chance level."""
    import torch
    opt = torch.optim.SGD(model.parameters(), lr=rc["lr_b"], momentum=0.9, weight_decay=5e-4,
                          nesterov=rc["nesterov_b"])
    scaler = torch.amp.GradScaler("cuda", enabled=a.amp)
    steps_per_epoch = max(1, len(train_set) // rc["batch"])
    total = max(1, rc["epochs_b"] * steps_per_epoch)
    warm = min(total, int(round(rc.get("warmup_b", 0.0) * steps_per_epoch)))

    def lr_fn(step):
        if step < warm:
            return rc["lr_b"] * (step + 1) / warm
        f = (step - warm) / max(1, total - warm)
        return 0.5 * rc["lr_b"] * (1.0 + math.cos(math.pi * min(f, 1.0)))

    start = 0
    if state and state.get("phase") == "B":
        opt.load_state_dict(state["opt"])
        scaler.load_state_dict(state["scaler"])
        start = state["epoch"]
    chance = math.log(lg["data"]["num_classes"])
    if start == 0:
        lg["phase_b_start_train_loss"] = start_train_loss(model, train_set, rc, a, dev)
        log("before fine-tune train loss %.4f (chance level ln %d = %.3f)"
            % (lg["phase_b_start_train_loss"], lg["data"]["num_classes"], chance))
    ref = lg.get("phase_b_start_train_loss")
    for e in range(start, rc["epochs_b"]):
        t0 = time.time()
        loss = train_epoch_dense(model, epoch_loader(train_set, rc, a, dev, "B", e), opt, dev,
                                 rc["label_smoothing"], rc["clip_b"], scaler, a.amp, lr_fn, e * steps_per_epoch)
        lg["history"].append({"phase": "B", "epoch": e + 1, "train_loss": loss, "seconds": time.time() - t0,
                              "lr_end": opt.param_groups[0]["lr"]})
        log("[B %d/%d] loss %.4f (%.0fs)" % (e + 1, rc["epochs_b"], loss, time.time() - t0))
        if not math.isfinite(loss) or (ref is not None and ref < 0.5 * chance and loss > 0.8 * chance):
            raise SystemExit("phase B collapsed: train loss %.3f is at chance level (ln %d = %.3f) although the "
                             "model entered fine-tuning with loss %s. Start this run again in a NEW --out folder "
                             "with a lower fine-tune learning rate for every method, e.g. --override lr_b=%g"
                             % (loss, lg["data"]["num_classes"], chance,
                                "%.3f" % ref if ref is not None else "n/a", rc["lr_b"] / 3))
        st = {"phase": "B", "epoch": e + 1, "model_sd": model.state_dict(), "arch": arch,
              "opt": opt.state_dict(), "scaler": scaler.state_dict(), "log": lg}
        st.update(extra or {})
        run.save(st)


def resume_state(a, run):
    if not a.resume:
        if os.path.exists(run.resume_path):
            raise SystemExit("%s exists: add --resume to continue that run, or use a new --out" % run.resume_path)
        return None
    st = run.load()
    if st is None:
        log("--resume: nothing to resume, starting from scratch")
    else:
        log("resuming phase %s after epoch %d" % (st["phase"], st["epoch"]))
    return st


def finish(a, run, model, arch, lg, eval_loader, dev, t_start):
    res = evaluate(model.to(dev), eval_loader, dev, a.amp)
    log("final %s top-1: %d/%d = %.2f%%, top-5 %.2f%%" % (lg["data"]["eval_split"], res["correct"], res["n"],
                                                           100 * res["top1"], 100 * res["top5"]))
    lg["final_eval"] = dict(res, split=lg["data"]["eval_split"])
    lg["model"] = complexity(model, a.input_size)
    lg["checkpoint"] = save_model(model, arch, os.path.join(run.out, "model.pt"))
    lg["wall_time_s"] = lg.get("wall_time_s", 0.0) + time.time() - t_start
    eb.write_json(os.path.join(run.out, "log.json"), lg)
    run.finish(a.keep_resume)
    log("wrote %s and %s" % (os.path.join(run.out, "model.pt"), os.path.join(run.out, "log.json")))



def cmd_baseline(a):
    t_start = time.time()
    rc, dev, train_set, eval_loader, info = setup(a, "baseline")
    run = Run(a.out)
    state = resume_state(a, run)
    if state:
        model, _ = eb.rebuild_from_state_dict(state["model_sd"], a.arch)
        lg = state["log"]
    else:
        model = new_model(a, info["num_classes"])
        lg = common_log(a, rc, info, dev, "baseline")
    model.to(dev)
    if not state or state["phase"] == "A":
        dense_phase_a(model, a, rc, dev, train_set, run, state, lg, a.arch)
        state = None
    phase_b(model, a, rc, dev, train_set, run, state, lg, a.arch)
    finish(a, run, model, a.arch, lg, eval_loader, dev, t_start)



def anneal(step, total, start, end, schedule="linear"):
    f = min(max(step / max(total, 1), 0.0), 1.0)
    return start * (end / start) ** f if schedule == "exp" else start + (end - start) * f


def build_gated(a, num_classes, pretrained):
    import timm
    base = timm.create_model(a.arch, pretrained=pretrained, num_classes=num_classes)
    g = GatedEdgeNeXt(base, a.arch, num_classes, a.input_size, a.gate_types.split(","), a.init_logit, a.sens_ema)
    measured = eb.count_macs(base, a.input_size)["macs"]
    if g.full_cost() != measured:
        raise RuntimeError("cost model %d != measured MACs %d" % (g.full_cost(), measured))
    return g


def cmd_gated(a):
    import torch
    import torch.nn.functional as F
    t_start = time.time()
    for t in a.gate_types.split(","):
        if t not in GATE_TYPES:
            raise SystemExit("unknown gate type %r (choose from %s)" % (t, ",".join(GATE_TYPES)))
    rc, dev, train_set, eval_loader, info = setup(a, "gated")
    run = Run(a.out)
    state = resume_state(a, run)
    nc = info["num_classes"]

    if state and state["phase"] == "B":
        lg = state["log"]
        pruned, _ = eb.rebuild_from_state_dict(state["model_sd"], a.arch)
        pruned.to(dev)
    else:
        g = build_gated(a, nc, pretrained=not a.no_pretrained and not state)
        full = g.full_cost()
        budget = rc["budget"] * full
        if state:
            g.load_state_dict(state["gated_sd"])
            lg = state["log"]
        else:
            lg = common_log(a, rc, info, dev, "gated")
            lg["gates"] = {"types": a.gate_types.split(","), "init_logit": a.init_logit, "sens_ema": a.sens_ema,
                           "groups": {k: int(v.numel()) for k, v in g.gates.items()}}
            lg["unpruned"] = complexity(g.model, a.input_size)
            lg["budget"] = {"fraction": rc["budget"], "unpruned_macs": full, "target_macs": budget}
            log("unpruned: %s params, %.2f M MACs; budget %.2f M MACs (%.0f%%)"
                % (f"{lg['unpruned']['params']:,}", full / 1e6, budget / 1e6, 100 * rc["budget"]))
        g.to(dev)
        opt, sched = make_opt_a([{"params": list(g.parameters_gates()), "lr": rc["gate_lr"], "weight_decay": 0.0},
                                 {"params": list(g.parameters_backbone()), "lr": rc["lr_a"],
                                  "weight_decay": rc["wd_a"]}], rc, rc["epochs_a"])
        scaler = torch.amp.GradScaler("cuda", enabled=a.amp)
        lam, step, start = 0.0, 0, 0
        rho = rc["rho_start"]
        n_batches = len(train_set) // rc["batch"]
        total = rc["epochs_a"] * n_batches
        if state:
            opt.load_state_dict(state["opt"])
            if sched and state.get("sched"):
                sched.load_state_dict(state["sched"])
            scaler.load_state_dict(state["scaler"])
            lam, step, start = state["lam"], state["step"], state["epoch"]

        for e in range(start, rc["epochs_a"]):
            t0 = time.time()
            g.train()
            apply_budget = e >= rc["warmup_epochs"]
            sums = {"task": 0.0, "cost": 0.0, "delta": 0.0, "n": 0}
            deltas = []
            for x, y in epoch_loader(train_set, rc, a, dev, "A", e):
                x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
                g.beta = anneal(step, total, rc["beta_start"], rc["beta_end"])
                rho = anneal(step, total, rc["rho_start"], rc["rho_end"], "exp")
                g.grad_scale = scaler.get_scale() if a.amp else 1.0
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=dev.type, enabled=a.amp):
                    task = F.cross_entropy(g(x), y, label_smoothing=rc["label_smoothing"])
                cost = g.soft_cost()
                delta = (cost - budget) / budget
                loss = task.float() + rc["alpha"] * g.sens_reg()
                if apply_budget:
                    loss = loss + lam * delta + 0.5 * rho * delta ** 2
                scaler.scale(loss).backward()
                if rc["clip_a"] > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_([p for grp in opt.param_groups for p in grp["params"]],
                                                   rc["clip_a"])
                scaler.step(opt)
                scaler.update()
                step += 1
                sums["task"] += float(task.detach())
                sums["cost"] += float(cost.detach())
                sums["n"] += 1
                if apply_budget:
                    deltas.append(float(delta.detach()))
            if apply_budget and deltas:
                lam = max(0.0, min(rc["lambda_max"], lam + rc["eta"] * sum(deltas) / len(deltas)))
            if sched:
                sched.step()
            hard = g.hard_cost()
            kept = g.hard_counts()
            rec = {"phase": "A", "epoch": e + 1, "budget_active": apply_budget,
                   "train_task_loss": sums["task"] / max(sums["n"], 1),
                   "soft_cost_macs": sums["cost"] / max(sums["n"], 1), "hard_cost_macs": hard,
                   "hard_cost_fraction": hard / full, "lambda": lam, "beta": g.beta, "rho": rho,
                   "kept_units": kept, "seconds": time.time() - t0}
            lg["history"].append(rec)
            log("[A %d/%d]%s loss %.4f  soft %.1fM  hard %.1fM (%.1f%% of unpruned, budget %.0f%%)  lambda %.3f  (%.0fs)"
                % (e + 1, rc["epochs_a"], "" if apply_budget else " warm-up", rec["train_task_loss"],
                   rec["soft_cost_macs"] / 1e6, hard / 1e6, 100 * hard / full, 100 * rc["budget"], lam,
                   rec["seconds"]))
            run.save({"phase": "A", "epoch": e + 1, "gated_sd": g.state_dict(), "opt": opt.state_dict(),
                      "sched": sched.state_dict() if sched else None, "scaler": scaler.state_dict(),
                      "lam": lam, "step": step, "log": lg})

        g.eval()
        before = g.hard_cost()
        kept_before = g.hard_counts()
        fit = g.fit_budget(budget if not a.no_enforce_budget else float("inf"), a.round_to)
        closed = fit["units_closed"]
        after = g.hard_cost()
        torch.save(g.state_dict()["gates"], os.path.join(run.out, "gates.pt"))
        pruned = export_pruned(g)
        xb = torch.stack([eval_loader.dataset[i][0] for i in range(min(8, len(eval_loader.dataset)))])
        g_cpu = g.to("cpu")
        diff = compare_outputs(g_cpu, pruned, xb)
        measured = eb.count_macs(pruned, a.input_size)["macs"]
        g.remove_hooks()
        lg["export"] = {"hard_cost_before_projection": before, "kept_units_before_projection": kept_before,
                        "projection": fit, "units_closed_by_budget_projection": closed,
                        "cost_model_macs": after, "measured_macs": measured,
                        "cost_model_equals_measured": after == measured,
                        "gated_vs_exported_max_abs_logit_diff": diff, "kept_units": g.hard_counts(),
                        "budget_met": measured <= budget + 0.5}
        lg["pruned_before_finetune"] = complexity(pruned, a.input_size)
        log("export: %s params, %.2f M MACs (%.1f%% of unpruned; %d units closed to meet budget); "
            "gated vs exported max |diff| %.2e" % (f"{lg['pruned_before_finetune']['params']:,}", measured / 1e6,
                                                     100 * measured / full, closed, diff))
        log("        (binary gates before rounding/projection: %.2f M MACs = %.1f%%; %d units opened by rounding "
            "to multiples of %d)" % (before / 1e6, 100 * before / full, fit["units_opened"], a.round_to))
        if after != measured:
            log("WARNING: cost model %d != measured %d" % (after, measured))
        if diff > 1e-3:
            log("WARNING: exported model differs from the gated model (max |diff| %.2e)" % diff)
        lg["export"]["checkpoint"] = save_model(pruned, a.arch, os.path.join(run.out, "exported_before_finetune.pt"))
        del g, g_cpu
        pruned.to(dev)
        pre = evaluate(pruned, eval_loader, dev, a.amp)
        lg["eval_before_finetune"] = dict(pre, split=info["eval_split"],
                                          note="reported only; not used for any decision")
        log("before fine-tune %s top-1: %.2f%%" % (info["eval_split"], 100 * pre["top1"]))
        state = None

    phase_b(pruned, a, rc, dev, train_set, run, state, lg, a.arch)
    finish(a, run, pruned, a.arch, lg, eval_loader, dev, t_start)



def _depgraph_ignored(model):
    """Every layer whose output channels are coupled to an SDTA-stage trunk or
    to XCA heads: Torch-Pruning cannot index those groups on EdgeNeXt (the
    IndexError), so they are kept intact. Prunable: MLP hidden units of every
    block and the stage-1 trunk."""
    ig = [model.head.fc]
    for s, st in enumerate(model.stages):
        if s >= 1 and len(st.downsample) > 1:
            ig.append(st.downsample[1])
        for b in st.blocks:
            if s >= 1:
                ig.append(b.mlp.fc2)
            if hasattr(b, "xca"):
                ig += [b.xca.qkv, b.xca.proj] + list(b.convs)
                if b.pos_embd is not None:
                    ig.append(b.pos_embd.token_projection)
    return ig


def depgraph_prune(model, ratio, input_size, round_to=None):
    import torch
    import torch.nn as nn
    import torch_pruning as tp
    m = copy.deepcopy(model).cpu().eval()
    ex = torch.randn(1, 3, input_size, input_size)
    imp_cls = getattr(tp.importance, "GroupMagnitudeImportance", None) or tp.importance.MagnitudeImportance
    kw = dict(importance=imp_cls(p=2), ignored_layers=_depgraph_ignored(m), root_module_types=[nn.Conv2d, nn.Linear])
    if round_to and round_to > 1:
        kw["round_to"] = round_to
    try:
        pruner = tp.pruner.MetaPruner(m, ex, pruning_ratio=ratio, **kw)
    except TypeError:
        pruner = tp.pruner.MetaPruner(m, ex, ch_sparsity=ratio, **kw)
    pruner.step()
    return m


def cmd_depgraph(a):
    import warnings
    t_start = time.time()
    try:
        import torch_pruning as tp
    except ImportError:
        raise SystemExit("pip install torch-pruning")
    rc, dev, train_set, eval_loader, info = setup(a, "depgraph")
    run = Run(a.out)
    state = resume_state(a, run)
    if state:
        model, _ = eb.rebuild_from_state_dict(state["model_sd"], a.arch)
        lg = state["log"]
    else:
        model = new_model(a, info["num_classes"])
        lg = common_log(a, rc, info, dev, "depgraph")
        lg["unpruned"] = complexity(model, a.input_size)
    model.to(dev)
    if not state or state["phase"] == "A":
        dense_phase_a(model, a, rc, dev, train_set, run, state, lg, a.arch)
        state = None
        full = lg["unpruned"]["macs"]
        target = rc["budget"] * full
        warnings.filterwarnings("ignore", message="Unwrapped parameters detected")
        grid = [round(x, 4) for x in np.arange(0.05, a.max_ratio + 1e-9, 0.05)]
        tried = []
        for r in grid:
            cand = depgraph_prune(model, r, a.input_size, a.round_to)
            tried.append((r, eb.count_macs(cand, a.input_size)["macs"], cand))
            if tried[-1][1] <= target:
                break
        feasible = [t for t in tried if t[1] <= target]
        if feasible:
            reached = True
            hi, best = feasible[0][0], feasible[0][2]
            lo = tried[-2][0] if len(tried) > 1 else 0.0
            for _ in range(a.bisect_steps):
                mid = (lo + hi) / 2
                cand = depgraph_prune(model, mid, a.input_size, a.round_to)
                if eb.count_macs(cand, a.input_size)["macs"] <= target:
                    hi, best = mid, cand
                else:
                    lo = mid
        else:
            reached = False
            hi, _, best = min(tried, key=lambda t: t[1])
        model = best
        macs = eb.count_macs(model, a.input_size)["macs"]
        lg["budget"] = {"fraction": rc["budget"], "unpruned_macs": full, "target_macs": target}
        lg["depgraph"] = {"importance": "GroupMagnitudeImportance(p=2)", "round_to": a.round_to,
                          "ratio": hi,
                          "budget_reachable": reached, "macs": macs,
                          "pruned_groups": "MLP hidden of every block + stage-1 trunk",
                          "kept_intact": "SDTA-stage trunks (stages 2-4), XCA heads, classifier"}
        lg["pruned_before_finetune"] = complexity(model, a.input_size)
        log("DepGraph: ratio %.3f -> %.2f M MACs (%.1f%% of unpruned)%s"
            % (lg["depgraph"]["ratio"], macs / 1e6, 100 * macs / full,
               "" if reached else "  BUDGET NOT REACHABLE with the prunable groups"))
        model.to(dev)
        pre = evaluate(model, eval_loader, dev, a.amp)
        lg["eval_before_finetune"] = dict(pre, split=info["eval_split"], note="reported only")
        log("before fine-tune %s top-1: %.2f%%" % (info["eval_split"], 100 * pre["top1"]))
    phase_b(model, a, rc, dev, train_set, run, state, lg, a.arch)
    finish(a, run, model, a.arch, lg, eval_loader, dev, t_start)



def _randomize(model, seed):
    """Non-trivial weights: LayerScale 1 instead of 1e-6, random biases and
    norm affine parameters, so a wrong index cannot hide."""
    import torch
    gen = torch.Generator().manual_seed(seed)
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name.endswith("gamma") or name.endswith("gamma_xca"):
                p.copy_(1.0 + 0.1 * torch.randn(p.shape, generator=gen))
            elif name.endswith("bias"):
                p.copy_(0.1 * torch.randn(p.shape, generator=gen))
            elif ("norm" in name or "stem.1" in name or "downsample.0" in name) and name.endswith("weight"):
                p.copy_(1.0 + 0.1 * torch.randn(p.shape, generator=gen))
            elif name.endswith("temperature"):
                p.copy_(0.5 + torch.rand(p.shape, generator=gen))


def cmd_check(a):
    import timm
    import torch
    eb.install_edgenext_patches()
    ok = True
    rows = []

    def record(name, value, passed):
        nonlocal ok
        ok = ok and passed
        rows.append((name, value, "PASS" if passed else "FAIL"))
        log("  %-62s %-14s %s" % (name, value, "PASS" if passed else "FAIL"))

    archs = a.archs.split(",")
    for arch in archs:
        for gate_types in (",".join(GATE_TYPES), "trunk,heads,mlp"):
            log("\n%s, gates: %s" % (arch, gate_types))
            torch.manual_seed(a.seed)
            base = timm.create_model(arch, pretrained=False, num_classes=a.num_classes)
            _randomize(base, a.seed)
            ref = copy.deepcopy(base).eval()
            g = GatedEdgeNeXt(base, arch, a.num_classes, a.input_size, gate_types.split(","))
            x = torch.randn(2, 3, a.input_size, a.input_size, generator=torch.Generator().manual_seed(1))
            g.eval()
            record("all gates open: gated == original (masked LayerNorm)", "%.1e" % compare_outputs(g, ref, x),
                   compare_outputs(g, ref, x) < 1e-4)
            measured = eb.count_macs(ref, a.input_size)["macs"]
            record("cost model (all open) == measured MACs", "%d" % g.full_cost(), g.full_cost() == measured)
            gen = torch.Generator().manual_seed(a.seed + 7)
            with torch.no_grad():
                for k, la in g.gates.items():
                    la.copy_(torch.randn(la.shape, generator=gen) + 0.4)
            pruned = export_pruned(g)
            d = compare_outputs(g, pruned, x)
            record("random gates: gated (eval) == exported", "%.1e" % d, d < 1e-4)
            mm = eb.count_macs(pruned, a.input_size)["macs"]
            record("cost model (binary gates) == measured MACs of exported", "%.2f%% of full" % (100 * mm / measured),
                   g.hard_cost() == mm)
            budget = 0.5 * measured
            fit = g.fit_budget(budget, 8)
            pruned2 = export_pruned(g)
            mm2 = eb.count_macs(pruned2, a.input_size)["macs"]
            record("rounding to x8 + projection: exported MACs <= 50% budget",
                   "%.2f%% (-%d/+%d)" % (100 * mm2 / measured, fit["units_closed"], fit["units_opened"]),
                   mm2 <= budget and g.hard_cost() == mm2)
            hc = g.hard_counts()
            aligned = all(v % 8 == 0 or v == g.gates[k].numel() for k, v in hc.items() if not k.startswith("head"))
            record("kept counts are multiples of 8 (or full)", "", aligned)
            d2 = compare_outputs(g, pruned2, x)
            record("after projection: gated (eval) == exported", "%.1e" % d2, d2 < 1e-4)
            rb, arch_inf = eb.rebuild_from_state_dict(pruned2.state_dict())
            d3 = compare_outputs(rb, pruned2, x)
            record("rebuild from state_dict == exported (arch inferred: %s)" % arch_inf, "%.1e" % d3,
                   d3 < 1e-6 and arch_inf == arch)
            summary = eb.arch_summary(pruned2)
            log("    exported: dims %s, heads %s, sdta split %s" % (summary["stage_dims"], summary["heads"],
                                                                   summary["sdta_split"]))
            g.train()
            g.beta = 0.5
            out = g(x)
            loss = out.float().logsumexp(1).mean() + 1e-9 * g.soft_cost()
            loss.backward()
            has_grad = all(p.grad is not None and torch.isfinite(p.grad).all() for p in g.parameters_gates())
            record("training step: finite gradients on every gate", "", bool(has_grad))
            record("Taylor sensitivities collected for every MLP gate", "%d" % len(g.sens),
                   len(g.sens) == sum(1 for k in g.gates if k.startswith("mlp")))
            g.remove_hooks()
            if a.onnx or a.tflite:
                import tempfile
                with tempfile.TemporaryDirectory() as td:
                    x1 = x[:1].numpy()
                    with torch.no_grad():
                        y1 = pruned2(x[:1]).numpy()
                    if a.onnx:
                        p = os.path.join(td, "m.onnx")
                        eb.export_onnx(pruned2, p, a.input_size, 17)
                        r = eb.OrtRunner(p, 1)
                        do = float(np.abs(eb.run_logits(r, x1) - y1).max())
                        record("ONNX Runtime == PyTorch (exported model)", "%.1e" % do, do < 1e-3)
                    if a.tflite:
                        p = os.path.join(td, "m.tflite")
                        eb.export_tflite_fp32(pruned2, p, a.input_size)
                        r = eb.TfliteRunner(p, 1)
                        dt = float(np.abs(eb.run_logits(r, x1) - y1).max())
                        record("TFLite FP32 == PyTorch (exported model)", "%.1e" % dt, dt < 1e-3)
    log("\n%s" % ("ALL CHECKS PASSED" if ok else "SOME CHECKS FAILED"))
    if not ok:
        sys.exit(1)



def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__.split("\n")[1],
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)

    def common(sp, budget):
        sp.add_argument("--arch", required=True, choices=sorted(set(eb.ARCH_BY_STAGE4_DIM.values())))
        sp.add_argument("--dataset", required=True, choices=list(eb.DATASETS))
        sp.add_argument("--data-root", required=True)
        sp.add_argument("--out", required=True, help="output folder (model.pt, log.json)")
        sp.add_argument("--recipe", choices=list(RECIPES),
                        help="default: xxs for edgenext_xx_small, s otherwise")
        sp.add_argument("--seed", type=int, default=0)
        sp.add_argument("--epochs-a", type=int, help="phase A epochs (recipe default)")
        sp.add_argument("--epochs-b", type=int, help="phase B epochs (recipe default)")
        sp.add_argument("--batch-size", type=int)
        sp.add_argument("--input-size", type=int, default=224)
        sp.add_argument("--class-order", help="tiny10/tiny200: comma-separated WNIDs in label order")
        sp.add_argument("--train-per-class", type=int, help="cap train images per class (smoke tests only)")
        sp.add_argument("--eval-per-class", type=int, help="cap eval images per class (smoke tests only)")
        sp.add_argument("--workers", type=int, default=2)
        sp.add_argument("--amp", action="store_true", help="mixed precision on CUDA (use for all methods or none)")
        sp.add_argument("--cpu", action="store_true")
        sp.add_argument("--no-pretrained", action="store_true")
        sp.add_argument("--download", action="store_true", help="CIFAR only")
        sp.add_argument("--resume", action="store_true")
        sp.add_argument("--keep-resume", action="store_true", help="keep resume.pt after the run")
        sp.add_argument("--override", nargs="*", metavar="KEY=VALUE", help="override recipe values")
        if budget:
            sp.add_argument("--budget", type=float, help="target MACs as a fraction of the unpruned model")

    b = sub.add_parser("baseline", help="unpruned model, same recipe")
    common(b, False)
    b.set_defaults(func=cmd_baseline)

    g = sub.add_parser("gated", help="proposed coupling-typed gated pruning")
    common(g, True)
    g.add_argument("--gate-types", default=",".join(GATE_TYPES),
                   help="comma list from %s; 'trunk,heads,mlp' = archived behaviour "
                        "(SDTA-stage trunks kept full)" % ",".join(GATE_TYPES))
    g.add_argument("--warmup-epochs", type=int, help="epochs without budget loss (recipe default)")
    g.add_argument("--init-logit", type=float, default=2.0)
    g.add_argument("--sens-ema", type=float, default=0.9)
    g.add_argument("--no-enforce-budget", action="store_true",
                   help="do not close extra gates if the binary gates exceed the budget")
    g.add_argument("--round-to", type=int, default=8,
                   help="kept channel / hidden-unit counts are multiples of this (1 = off). Unaligned "
                        "widths can erase the latency gain on CPU runtimes")
    g.set_defaults(func=cmd_gated)

    d = sub.add_parser("depgraph", help="Torch-Pruning baseline on the groups it can handle")
    common(d, True)
    d.add_argument("--max-ratio", type=float, default=0.95)
    d.add_argument("--round-to", type=int, default=8, help="Torch-Pruning round_to (1 = off)")
    d.add_argument("--bisect-steps", type=int, default=8)
    d.set_defaults(func=cmd_depgraph)

    c = sub.add_parser("check", help="self-test, no data needed")
    c.add_argument("--archs", default="edgenext_xx_small,edgenext_small")
    c.add_argument("--num-classes", type=int, default=10)
    c.add_argument("--input-size", type=int, default=224)
    c.add_argument("--seed", type=int, default=0)
    c.add_argument("--onnx", action="store_true", help="also check ONNX Runtime parity")
    c.add_argument("--tflite", action="store_true", help="also check TFLite FP32 parity")
    c.set_defaults(func=cmd_check)

    args = p.parse_args(argv)
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
!pip -q install timm==1.0.30 torch-pruning==1.6.1
!python -m py_compile edgenext_bench.py edgenext_gated_prune.py && echo "scripts OK"
!python edgenext_gated_prune.py check --archs {ARCH}


In [ ]:
DS  = {'cifar10': 'c10', 'cifar100': 'c100', 'tiny200': 't200', 'imagefolder': 'in1k'}[DATASET]
MD  = {'edgenext_xx_small': 'xxs', 'edgenext_small': 's', 'edgenext_x_small': 'xs'}[ARCH]
B   = int(round(BUDGET * 100))
RUN = f'{WORK}/runs/{DS}_{MD}'
os.makedirs('/kaggle/temp', exist_ok=True)
if DATASET in ('cifar10', 'cifar100'):
    DATA_ROOT = '/kaggle/temp/data'
    os.makedirs(DATA_ROOT, exist_ok=True)
    import torchvision
    C = torchvision.datasets.CIFAR10 if DATASET == 'cifar10' else torchvision.datasets.CIFAR100
    C(DATA_ROOT, train=True, download=True); C(DATA_ROOT, train=False, download=True)
elif DATASET == 'tiny200':
    found = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/wnids.txt', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'val', 'val_annotations.txt'))]
    if found:
        DATA_ROOT = found[0]
    else:
        DATA_ROOT = '/kaggle/temp/tiny-imagenet-200'
        if not os.path.exists(f'{DATA_ROOT}/wnids.txt'):
            import zipfile
            z = '/kaggle/temp/tiny.zip'
            for url in ('https://cs231n.stanford.edu/tiny-imagenet-200.zip',
                        'http://cs231n.stanford.edu/tiny-imagenet-200.zip'):
                if os.path.exists(z):
                    os.remove(z)
                r = subprocess.run(['wget', '-q', '--tries=3', '--timeout=60', '-O', z, url])
                if r.returncode == 0 and zipfile.is_zipfile(z):
                    subprocess.run(['unzip', '-q', '-o', z, '-d', '/kaggle/temp'])
                    os.remove(z)
                    break
                print('download failed from', url)
            if os.path.exists(z):
                os.remove(z)
        if not os.path.exists(f'{DATA_ROOT}/wnids.txt'):
            raise SystemExit('Tiny-ImageNet is not available. Add Input -> Datasets -> search "tiny-imagenet-200", '
                             'add one that contains wnids.txt, train/ and val/val_annotations.txt, then run this cell again.')
else:
    DATA_ROOT = '/kaggle/temp/imagenet'
print(DATASET, ARCH, SEEDS, BUDGET, 'data:', DATA_ROOT, '| runs:', RUN)
!ls {DATA_ROOT} | head


In [ ]:
def out_dir(method, seed):
    if method == 'baseline':
        return f'{RUN}/baseline_s{seed}'
    return f'{RUN}/{method}{TAG if method == "gated" else ""}_b{B}_s{seed}'

if DO_TRAIN:
    for seed in SEEDS:
        for method in METHODS:
            out = out_dir(method, seed)
            if os.path.exists(f'{out}/log.json'):
                print('done already:', out); continue
            budget = '' if method == 'baseline' else f'--budget {BUDGET}'
            extra = EXTRA_GATED if method == 'gated' else ''
            print('\n=== running', out)
            !python edgenext_gated_prune.py {method} --arch {ARCH} --dataset {DATASET} --data-root {DATA_ROOT} \
                {budget} --seed {seed} {AMP} {extra} --workers {WORKERS} --resume --out {out}

In [ ]:
for seed in SEEDS:
    for method in METHODS:
        p = out_dir(method, seed) + '/log.json'
        if not os.path.exists(p):
            print(p, 'not finished'); continue
        L = json.load(open(p))
        line = (f"{os.path.basename(os.path.dirname(p))}: top-1 {100*L['final_eval']['top1']:.2f}% on "
                f"{L['final_eval']['n']} images, {L['model']['params']:,} params, {L['model']['macs']/1e6:.1f} M MACs, "
                f"{L['data']['n_train']} train images, {L.get('wall_time_s', 0)/3600:.2f} h")
        if 'export' in L:
            e = L['export']
            line += (f" | cost==measured {e['cost_model_equals_measured']}, gated-vs-exported "
                     f"{e['gated_vs_exported_max_abs_logit_diff']:.1e}, budget met {e['budget_met']}, "
                     f"closed by projection {e['units_closed_by_budget_projection']}")
        if 'depgraph' in L:
            line += f" | budget reachable {L['depgraph']['budget_reachable']}"
        if 'eval_before_finetune' in L:
            line += f" | before fine-tune {100*L['eval_before_finetune']['top1']:.2f}%"
        b = [h['train_loss'] for h in L['history'] if h['phase'] == 'B']
        if b:
            line += f" | fine-tune loss first/last {b[0]:.3f}/{b[-1]:.3f}"
            ref = L.get('phase_b_start_train_loss')
            if b[0] > 0.8 * math.log(L['data']['num_classes']) and (ref is None or ref < 0.5 * math.log(L['data']['num_classes'])):
                line += '  <-- COLLAPSED, rerun'
        print(line)

In [ ]:
EXPORT_ENV = '/kaggle/temp/exportenv'
EXPORT_PY = f'{EXPORT_ENV}/bin/python'
PINS = ('timm==1.0.30 onnx==1.23.2 onnxruntime==1.29.0 ai-edge-litert==2.3.0 litert-torch==0.10.0 '
        'ai-edge-quantizer==0.10.0 numpy pillow')
if DO_EXPORT and not os.path.exists(EXPORT_PY):
    r = subprocess.run(['python', '-m', 'venv', EXPORT_ENV])
    if r.returncode != 0 or not os.path.exists(EXPORT_PY):
        !pip -q install virtualenv && python -m virtualenv -q {EXPORT_ENV}
    !{EXPORT_PY} -m pip -q install --upgrade pip
    r = subprocess.run([EXPORT_PY, '-m', 'pip', '-q', 'install', 'torch==2.14.1', 'torchvision==0.29.1',
                        '--index-url', 'https://download.pytorch.org/whl/cpu'])
    if r.returncode != 0:
        !{EXPORT_PY} -m pip -q install torch==2.14.1 torchvision==0.29.1
    !{EXPORT_PY} -m pip -q install {PINS}
    !{EXPORT_PY} -c "import torch, timm, onnxruntime, litert_torch, ai_edge_quantizer; print('export env OK, torch', torch.__version__)"

if DO_EXPORT:
    extra = {'imagefolder': '--eval-per-class 10', 'cifar10': '--download', 'cifar100': '--download'}.get(DATASET, '')
    for seed in SEEDS:
        base = out_dir('baseline', seed) + '/model.pt'
        for method in [m for m in METHODS if m != 'baseline']:
            pruned = out_dir(method, seed) + '/model.pt'
            exp = os.path.basename(os.path.dirname(os.path.dirname(pruned))) + '_' + os.path.basename(os.path.dirname(pruned))
            dest = f'{WORK}/artifacts/{exp}'
            if not (os.path.exists(pruned) and os.path.exists(base)):
                print('skip', exp, '(pruned or baseline model missing)'); continue
            if os.path.exists(f'{dest}/manifest.json'):
                print('packaged already:', exp); continue
            for old in (dest, dest + '.zip'):
                if os.path.isdir(old): shutil.rmtree(old)
                elif os.path.exists(old): os.remove(old)
            !TQDM_DISABLE=1 {EXPORT_PY} edgenext_bench.py export --experiment {exp} --dataset {DATASET} \
                --data-root {DATA_ROOT} --pruned {pruned} --unpruned {base} --out {dest} {extra}
            man = f'{dest}/manifest.json'
            print(exp, (json.load(open(man))['warnings'] or 'no warnings') if os.path.exists(man) else 'EXPORT FAILED - see above')
    for d in sorted(glob.glob(f'{WORK}/artifacts/*/')):
        d = d.rstrip('/')
        man = d + '/manifest.json'
        if os.path.exists(man) and (not os.path.exists(d + '.zip') or os.path.getmtime(d + '.zip') < os.path.getmtime(man)):
            shutil.make_archive(d, 'zip', os.path.dirname(d), os.path.basename(d))
    !ls -lh {WORK}/artifacts/*.zip 2>/dev/null

if os.path.isdir(RUN):
    shutil.make_archive(f'{WORK}/runs_{DS}_{MD}', 'zip', f'{WORK}/runs', os.path.basename(RUN))
    print('run folders:', f'{WORK}/runs_{DS}_{MD}.zip', round(os.path.getsize(f'{WORK}/runs_{DS}_{MD}.zip') / 2**20, 1), 'MiB')

In [ ]:
!rm -rf /kaggle/working/edgenext/__pycache__
